# Image Editing LLM Pipeline — CS668 Applied LLMs
**Spec version:** 1.7 · **Notebook version:** 2.10 · **Target runtime:** Google Colab A100  
**Phases in this notebook:** Phase 0 — Dataset Audit & Formatting  
**Version:** v2.10 — §7.3 clean-subset filter added; sequential first-N sampling of tar.split files)
*(Phase 1 VLM training, Phase 2 diffusion bridge, Phase 3 inference, Phase 4 eval follow in separate notebooks or later cells)*

---
### Restart-safety contract
Every expensive cell saves its output to Google Drive **before returning**.  
Any cell can be re-run without corrupting prior outputs — re-runs detect existing artifacts and skip or resume.  
**Never** assume the runtime survived from a previous session.


## §0 — Global Configuration
*Change values here; never hardcode paths or hyperparameters elsewhere.*

In [ ]:
# ── §0.1  GLOBAL CONFIG — edit this cell only ──────────────────────────────
# All downstream cells import from CFG; never hardcode paths elsewhere.

import os, sys
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class PipelineConfig:
    # ── Drive root ────────────────────────────────────────────────────────
    drive_root: Path = Path("/content/drive/MyDrive/img_edit_pipeline")

    # ── Dataset ───────────────────────────────────────────────────────────
    hf_dataset_id: str         = "sysuyy/ImgEdit"
    # Subset of singleturn parquet configs to stream (None = all)
    hf_configs: Optional[list] = None   # e.g. ["remove", "add"] for partial pull
    # ⚠ DISK CONSTRAINT: each tar split ≈ 42.9 GB (confirmed §2.6).
    # With streaming extraction (v2.9), peak disk = 1 split in /tmp (~43 GB)
    # + extracted JPEGs on Drive (~0.1 MB each).
    # Sequential first-N sampling: take first n_subset rows from parquets in
    # file order — images concentrate in earliest tar splits, so streaming
    # terminates early rather than scanning the entire archive.
    # 10_000 samples → fast smoke/train cycle for a course project.
    # 50_000 samples → ~10× longer streaming; requires stable Colab connection.
    n_subset: int              = 10_000   # increase to 50_000 for full run

    # ── Edit types the pipeline must handle ───────────────────────────────
    # Filled in after dataset inspection; listed here as the expected set.
    # audit_dataset() will validate actuals match.
    # Confirmed from §2.1 output — parquet filename prefixes.
    # Note: "action","content","hybrid","reference","version" are NOT in the
    # original spec but exist in the actual dataset. Updated here.
    # Multi-turn types appear in sysuyy/ImgEdit_recap_mask under separate configs.
    expected_edit_types: tuple = (
        # confirmed singleturn (from §2.1 parquet filenames)
        "action", "add", "adjust", "background", "content",
        "hybrid", "reference", "remove", "replace", "style", "version",
    )

    # Mask dataset ID — contains RLE masks; main dataset has NO segmentation field.
    mask_dataset_id: str = "sysuyy/ImgEdit_recap_mask"

    # ── Bbox routing — keywords for global-adjust classification ─────────
    global_adjust_keywords: tuple = (
        "lighting", "light", "brightness", "exposure", "contrast",
        "saturation", "hue", "tone", "tones", "overall", "scene",
        "atmosphere", "color temperature", "white balance",
    )

    # ── Phase 1 model ─────────────────────────────────────────────────────
    vlm_model_id: str  = "Qwen/Qwen2.5-VL-3B-Instruct"
    vlm_quant_bits: int = 4
    vlm_lora_r: int    = 16
    vlm_lora_alpha: int = 32
    vlm_lora_dropout: float = 0.05
    vlm_hidden_dim: int = 2048    # Qwen2.5-VL-3B hidden size — verified at load time

    # ── Image archive handling ────────────────────────────────────────────
    # Temporary directory for split-tar reassembly and extraction.
    # Reassembled tars are DELETED immediately after image extraction to save disk.
    # Peak disk usage = 1 reassembled tar at a time (up to ~43 GB) + extracted JPEGs.
    # Confirmed §2.6: one tar stem (results_remove_laion_part1) ≈ 43 GB.
    # With 235 GB available, process-and-delete is required.
    image_tar_cache_dir: Path = None   # None → drive_root / "cache" / "image_tars_tmp"
    keep_image_tars: bool = False      # False = delete reassembled .tar after extraction
                                       # True  = keep cached (only if you have >>200 GB free)

    # ── Phase 2 model ─────────────────────────────────────────────────────
    sd_model_id: str           = "runwayml/stable-diffusion-inpainting"
    sd_cross_attn_dim: int     = 768   # SD 1.5 cross-attention dim (constant)
    sd_lora_r: int             = 8
    sd_lora_alpha: int         = 16

    # ── Training ──────────────────────────────────────────────────────────
    phase1_epochs: int         = 3
    phase1_batch_size: int     = 2
    phase1_grad_accum: int     = 8
    phase1_lr: float           = 2e-4
    phase1_max_seq_len: int    = 1024
    phase1_warmup_steps: int   = 100
    phase1_eval_steps: int     = 200

    phase2_epochs: int         = 5
    phase2_batch_size: int     = 1
    phase2_grad_accum: int     = 16
    phase2_lr: float           = 1e-4
    phase2_warmup_steps: int   = 200
    phase2_shard_size: int     = 5_000   # samples per hidden-state shard file

    # ── Audit thresholds (gates before training) ──────────────────────────
    max_invalid_rle_frac: float    = 0.05   # >5 % → inspect before Phase 2
    max_fallback_frac: float       = 0.15   # >15 % per type → inspect before Phase 1

    # ── Derived paths (computed, do not edit) ─────────────────────────────
    @property
    def data_dir(self) -> Path:
        return self.drive_root / "data" / "imgedit_subset"

    @property
    def benchmark_dir(self) -> Path:
        return self.drive_root / "data" / "benchmark"

    @property
    def ckpt_phase1(self) -> Path:
        return self.drive_root / "checkpoints" / "phase1_vlm"

    @property
    def ckpt_phase2(self) -> Path:
        return self.drive_root / "checkpoints" / "phase2_diffusion"

    @property
    def outputs_eval(self) -> Path:
        return self.drive_root / "outputs" / "eval"

    @property
    def outputs_scores(self) -> Path:
        return self.drive_root / "outputs" / "scores"

    @property
    def manifest_path(self) -> Path:
        return self.data_dir / "samples.json"

    @property
    def audit_path(self) -> Path:
        return self.data_dir / "audit_report.json"


CFG = PipelineConfig()
print("Config loaded.")
print(f"  Drive root    : {CFG.drive_root}")
print(f"  Data dir      : {CFG.data_dir}")
print(f"  n_subset      : {CFG.n_subset:,}")
print(f"  VLM model     : {CFG.vlm_model_id}")
print(f"  SD model      : {CFG.sd_model_id}")


## §0.2 — Google Drive Mount & Directory Scaffold

In [ ]:
# ── §0.2  Mount Drive and create directory tree ────────────────────────────
# Re-runnable: mounting an already-mounted Drive is a no-op.

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def scaffold_dirs(cfg) -> None:
    """Create all required Drive directories. Idempotent."""
    dirs = [
        cfg.data_dir,
        cfg.benchmark_dir,
        cfg.ckpt_phase1,
        cfg.ckpt_phase2,
        cfg.outputs_eval,
        cfg.outputs_scores,
    ]
    for d in dirs:
        d.mkdir(parents=True, exist_ok=True)
    print(f"Drive scaffold verified ({len(dirs)} directories).")
    for d in dirs:
        print(f"  ✓  {d}")

scaffold_dirs(CFG)


## §1 — Environment: Package Installation
⚠ **Run once per runtime.** After this cell finishes, Colab will prompt to restart the kernel. **Restart, then continue from §1.2.**  
Exact versions are pinned — do not loosen constraints without updating this spec comment.


In [ ]:
# ── §1.1  Install pinned dependencies ─────────────────────────────────────
# Pin reasons are documented inline. Do not change versions silently.
#
# transformers>=4.49.0  — first release with native Qwen2.5-VL support
#                         (Qwen2_5_VLForConditionalGeneration in core)
# peft==0.12.0          — LoRA API stable; 0.13+ changed get_peft_model return
# trl==0.11.0           — SFTTrainer data_collator signature stable
# bitsandbytes==0.44.0  — 4-bit NF4 + bfloat16 compute tested on Colab A100
# diffusers==0.30.0     — runwayml/stable-diffusion-inpainting confirmed path
# datasets==2.21.0      — streaming shuffle + .take() API stable
# SAM2 install notes (SPEC DEVIATION — see comments inside cell):
#   Repo renamed to facebookresearch/sam2; no release tags exist; cuda ext skipped.

import os, subprocess, sys

def pip_install(packages: list[str], flags: str = "") -> None:
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + packages
    if flags:
        cmd += flags.split()
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError(f"pip install failed: {packages}")
    print(f"  installed: {packages}")

print("Installing core packages...")
pip_install([
    "transformers==4.49.0", # Updated from 4.46.0
    "peft==0.12.0",
    "trl==0.11.0",
    "bitsandbytes==0.44.0",
    "accelerate==0.34.0",
    "diffusers==0.30.0",
    "datasets==2.21.0",
    "safetensors",
    "openai",
    "tqdm",
    "tenacity",
    "lpips",
    "pycocotools",
])

print("Installing hydra-core (SAM2 dependency)...")
pip_install(["hydra-core>=1.3.2"])

print("Installing SAM2 from facebookresearch/sam2 ...")
# SPEC DEVIATION — documented:
#   Original spec: "segment-anything-2 @v1.0" — tag never existed;
#   repo renamed to facebookresearch/sam2; no formal tags in repo.
# SAM2_BUILD_CUDA=0 skips optional CUDA extension (sam2._C). It provides
# only connected-component post-processing; SAM2ImagePredictor works without it.
# torch>=2.5.1 required by current sam2 main (spec cited 2.3.1 — incorrect).
# After first run, copy the printed commit hash into SAM2_COMMIT_HASH to lock.
SAM2_COMMIT_HASH = ""  # set to 40-char SHA after first successful install
sam2_ref = f"@{SAM2_COMMIT_HASH}" if SAM2_COMMIT_HASH.strip() else ""
sam2_url = f"git+https://github.com/facebookresearch/sam2.git{sam2_ref}"
print(f"  URL: {sam2_url}")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", sam2_url],
    capture_output=True, text=True,
    env={**os.environ, "SAM2_BUILD_CUDA": "0"},
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])
    raise RuntimeError(
        "SAM2 install failed.\n"
        "Check internet access and that SAM2_BUILD_CUDA=0 env was set.\n"
        f"URL: {sam2_url}"
    )

# Print installed commit hash — copy to SAM2_COMMIT_HASH above to lock version
try:
    r = subprocess.run(["pip", "show", "SAM-2"], capture_output=True, text=True)
    for ln in r.stdout.splitlines():
        if ln.startswith(("Name", "Version", "Location")):
            print(f"  {ln}")
except Exception:
    pass
print("  installed: SAM2 from facebookresearch/sam2")

print()
print("=" * 60)
print("DONE. Restart the Colab runtime now (Runtime → Restart runtime).")
print("Then continue from §1.2 — do NOT re-run this cell.")
print("=" * 60)


## §1.2 — Import & Version Sanity Checks
*Run this cell after restarting runtime. It will raise on any version mismatch.*

In [ ]:
# ── §1.2  Import and version gate ─────────────────────────────────────────
# Fail fast with a clear message rather than discovering version issues
# mid-training. All assertions must pass before any downstream cell runs.

import importlib, importlib.metadata, sys
from packaging.version import Version

REQUIRED = {
    "torch":          ("2.5.1",  None),   # SAM2 main requires >=2.5.1 (spec cited 2.3.1 for @v1.0 which does not exist)
    "transformers":   ("4.49.0", None),   # Updated from 4.46.0 due to Qwen2_5_VLForConditionalGeneration import issue
    "peft":           ("0.12.0", "0.13.0"),
    "trl":            ("0.11.0", "0.12.0"),
    "bitsandbytes":   ("0.44.0", None),
    "accelerate":     ("0.34.0", None),
    "diffusers":      ("0.30.0", "0.31.0"),
    "datasets":       ("2.21.0", None),
    "pycocotools":    ("2.0.0",  None),
}

failures = []
for pkg, (vmin, vmax) in REQUIRED.items():
    try:
        try:
            ver = Version(importlib.metadata.version(pkg))
        except importlib.metadata.PackageNotFoundError:
            mod = importlib.import_module(pkg)
            ver = Version(getattr(mod, "__version__", "0.0.0"))
        if ver < Version(vmin):
            failures.append(f"  ✗  {pkg} {ver} < required {vmin}")
        elif vmax and ver >= Version(vmax):
            failures.append(f"  ✗  {pkg} {ver} >= ceiling {vmax} (API may have changed)")
        else:
            print(f"  ✓  {pkg} {ver}")
    except ImportError as e:
        failures.append(f"  ✗  {pkg} not importable: {e}")

if failures:
    raise EnvironmentError("Version check FAILED:\n" + "\n".join(failures))

# ── CUDA ──────────────────────────────────────────────────────────────────
import torch
if not torch.cuda.is_available():
    raise EnvironmentError("CUDA not available — switch runtime to GPU (A100).")
print(f"  ✓  CUDA {torch.version.cuda} | device: {torch.cuda.get_device_name(0)}")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"     VRAM: {vram_gb:.1f} GB  (need ≥24 GB for Phase 1, ≥18 GB for Phase 2)")
if vram_gb < 18:
    raise EnvironmentError(f"Insufficient VRAM: {vram_gb:.1f} GB. Need ≥18 GB.")

# ── Qwen2.5-VL native class ───────────────────────────────────────────────
try:
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    print("  ✓  Qwen2_5_VLForConditionalGeneration importable from transformers")
except ImportError as e:
    raise EnvironmentError(
        f"Qwen2.5-VL class not in transformers — need >=4.49.0. Error: {e}" # Updated required version in error message
    )

# ── SAM2 ─────────────────────────────────────────────────────────────────
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    print("  ✓  sam2 importable (build_sam2, SAM2ImagePredictor)")
except ImportError as e:
    raise EnvironmentError(
        f"SAM2 not importable. Run §1.1 install cell first. Error: {e}"
    )

# ── Diffusers pipeline class ───────────────────────────────────────────────
from diffusers import StableDiffusionInpaintPipeline
print("  ✓  StableDiffusionInpaintPipeline importable")

# ── pycocotools RLE ───────────────────────────────────────────────────────
import pycocotools.mask as mask_utils
print("  ✓  pycocotools.mask importable")

# ── Re-import CFG (config must be reloaded after kernel restart) ───────────
# NOTE: After restart, re-run the §0 config cell before this cell.
try:
    _ = CFG
    print("  ✓  CFG in scope")
except NameError:
    raise RuntimeError("CFG not defined — re-run §0 config cell first.")

print()
print("All checks passed. Environment is ready.")


## §2 — Dataset Schema Inspection
**Run this before writing any data-loading code.** The spec names fields based on the README;  
this cell confirms actual parquet columns and prints raw sample structure.  
All subsequent code uses only fields confirmed here.


In [ ]:
import re
import numpy as np
from huggingface_hub import HfApi, hf_hub_download
from datasets import load_dataset
import pyarrow.parquet as pq
from pprint import pformat

HF_MAIN_ID   = CFG.hf_dataset_id          # "sysuyy/ImgEdit"
HF_MASK_ID   = CFG.mask_dataset_id        # "sysuyy/ImgEdit_recap_mask"
INSPECT_FILE = None   # set to a filename like "Parquet/add_part0.parquet" to re-inspect

api = HfApi()

# ── Helpers ────────────────────────────────────────────────────────────────
KNOWN_EDIT_TYPES = set(CFG.expected_edit_types)

def infer_edit_type_from_filename(fname: str) -> str:
    """Extract edit type from parquet filename prefix.

    Pattern: Parquet/{edit_type}[_{qualifier}]_part{N}.parquet
    Examples:
        background_part0.parquet      → background
        adjust_canny_part0.parquet    → adjust
        content_memory_part2.parquet  → content
    Strategy: first _-separated token(s) that match a known edit type.
    """
    stem = fname.split("/")[-1].replace(".parquet", "")  # strip dir + ext
    tokens = stem.split("_")
    accumulated = ""
    for tok in tokens:
        if tok.startswith("part") and tok[4:].isdigit():
            break
        accumulated = (accumulated + "_" + tok).lstrip("_")
        if accumulated in KNOWN_EDIT_TYPES:
            return accumulated
    return tokens[0] if tokens else "unknown"


def summarize_value(val, max_str_len: int = 120) -> str:
    """Compact, safe summary for schema inspection output."""
    if isinstance(val, str):
        return f"str = {val[:max_str_len]!r}" + ("..." if len(val) > max_str_len else "")
    if isinstance(val, list):
        preview = val[0] if len(val) else None
        return f"list[{len(val)}] first={preview!r}"
    if isinstance(val, tuple):
        preview = val[0] if len(val) else None
        return f"tuple[{len(val)}] first={preview!r}"
    if isinstance(val, np.ndarray):
        preview = val[0] if val.size else None
        return f"ndarray shape={val.shape} dtype={val.dtype} first={preview!r}"
    return f"{type(val).__name__} = {val!r}"


def extract_first_string(x):
    """
    Robustly extract a single filename/path string from a field that may be:
      - str
      - list/tuple of str
      - numpy.ndarray of str/object
      - nested single-item containers
    Returns None if no string can be extracted.
    """
    if x is None:
        return None

    if isinstance(x, str):
        return x

    if isinstance(x, np.ndarray):
        if x.size == 0:
            return None
        return extract_first_string(x.flat[0])

    if isinstance(x, (list, tuple)):
        if len(x) == 0:
            return None
        return extract_first_string(x[0])

    return str(x)


def parse_sample_id_from_path(sample_path: str) -> str:
    """
    Try to recover a sample id from a path-like string.

    Expected older pattern:
        results_{edit_type}_laion_part{N}/{sample_id}/{original|result}.png
    In that case sample_id is path_parts[1].

    If only a bare filename is present, return the stem as a fallback:
        hOdCsLRNAyw_segment_21_frame_0.jpg
        -> hOdCsLRNAyw_segment_21_frame_0
    """
    if not sample_path:
        return "UNKNOWN"

    norm = str(sample_path).replace("\\", "/")
    path_parts = [p for p in norm.split("/") if p]

    if len(path_parts) >= 3:
        return path_parts[1]

    filename = path_parts[-1] if path_parts else norm
    stem = re.sub(r"\.[^.]+$", "", filename)
    return stem if stem else "UNKNOWN"


# ── List parquet files ─────────────────────────────────────────────────────
print(f"Listing files in {HF_MAIN_ID} ...")
all_files = api.list_repo_files(HF_MAIN_ID, repo_type="dataset")
parquet_files = sorted([f for f in all_files if f.endswith(".parquet")])

print(f"Found {len(parquet_files)} parquet files:")
for f in parquet_files[:20]:
    print(f"  {f}")
if len(parquet_files) > 20:
    print(f"  ... and {len(parquet_files) - 20} more")

# ── Infer edit types from filenames ────────────────────────────────────────
edit_types_found = sorted(set(infer_edit_type_from_filename(f) for f in parquet_files))
print(f"\nEdit types inferred from filenames: {edit_types_found}")

unknown = [f for f in parquet_files if infer_edit_type_from_filename(f) == "unknown"]
if unknown:
    print("WARNING — could not infer edit type for:")
    for f in unknown:
        print(f"  {f}")

# ── Inspect one parquet file ───────────────────────────────────────────────
inspect_target = INSPECT_FILE or parquet_files[0]
print(f"\nInspecting: {inspect_target}")

local_pq = hf_hub_download(
    repo_id=HF_MAIN_ID,
    repo_type="dataset",
    filename=inspect_target
)

table = pq.read_table(local_pq)

print(f"\n{'=' * 60}")
print(f"SCHEMA from {inspect_target}:")
print(f"{'=' * 60}")
print(table.schema)
print(f"\nRow count: {len(table):,}")

df = table.to_pandas()

print(f"\n{'=' * 60}")
print("RAW SAMPLE[0]:")
print(f"{'=' * 60}")

row = df.iloc[0]
for col in df.columns:
    val = row[col]
    print(f"  {col!r}: {summarize_value(val)}")

# ── Extract sample_id from image path robustly ─────────────────────────────
raw_input_images = df.iloc[0]["input_images"]
print(f"\nRaw input_images type: {type(raw_input_images).__name__}")
print(f"Raw input_images value: {raw_input_images!r}")

sample_path = extract_first_string(raw_input_images)
print(f"\nSample image path: {sample_path!r}")

sample_id_candidate = parse_sample_id_from_path(sample_path)
print(f"Sample ID candidate: {sample_id_candidate!r}")
print(f"  (Will be used as join key with {HF_MASK_ID})")

# Optional diagnostic: show how the parser interpreted the path
if sample_path is not None:
    norm_path = str(sample_path).replace("\\", "/")
    print(f"Normalized path parts: {norm_path.split('/')}")

# ── List mask dataset files ────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"Listing files in mask dataset: {HF_MASK_ID}")
print(f"{'=' * 60}")

try:
    mask_files = sorted(api.list_repo_files(HF_MASK_ID, repo_type="dataset"))
    print(f"Found {len(mask_files)} files:")
    for f in mask_files[:15]:
        print(f"  {f}")
    if len(mask_files) > 15:
        print(f"  ... and {len(mask_files) - 15} more")

    mask_parquets = [f for f in mask_files if f.endswith(".parquet")]
    print(f"  Parquet files: {len(mask_parquets)}")
except Exception as e:
    print(f"  Error listing mask dataset: {e}")
    mask_parquets = []

print("\n§2.1 done. Confirmed schema recorded in §2.2.")

In [ ]:
# ── §2.2  Schema contract — CONFIRMED from §2.1 output ────────────────────
# This cell records the confirmed field mapping for all downstream cells.
# DO NOT change values on the right side — downstream code imports from here.

# ── Confirmed: main dataset (sysuyy/ImgEdit) ──────────────────────────────
# Source: §2.1 output on Parquet/background_part0.parquet
#
# Parquet columns (exactly 3 — confirmed):
#   input_images  : list[str]  — path(s) into tar archive, e.g.
#                     "results_background_laion_part0/00121_00035_000359496/original.png"
#   output_images : list[str]  — same pattern, "result.png" variant
#   prompt        : str        — natural-language edit instruction
#
# Images are STRING PATHS, not embedded pixels. They live in tar archives
# in the HuggingFace repo. Access via HfFileSystem or hf_hub_download.
#
# edit_type is NOT a dataset field. It is inferred from the parquet filename:
#   Parquet/{edit_type}[_{qualifier}]_part{N}.parquet → first token = edit_type
#   e.g.  adjust_canny_part0.parquet → "adjust"
#
# sample_id is NOT a dataset field. It is extracted from image path[1]:
#   "results_background_laion_part0/00121_00035_000359496/original.png"
#    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ part_dir  ^^^^^^^^^^^^^^^^^ sample_id
#
# Segmentation / masks: NOT PRESENT in this dataset.
#   Source: sysuyy/ImgEdit_recap_mask (separate HuggingFace dataset).
#   Join key: sample_id (extracted from image path).

# ── Confirmed: edit types (from parquet filenames, §2.1) ──────────────────
# NOT matching spec §4.2 — actual dataset has more types than spec listed.
# Spec expected: add, remove, replace, adjust, style, background + multi-turn
# Actual (confirmed): action, add, adjust, background, content, hybrid,
#                     reference, remove, replace, style, version
CONFIRMED_EDIT_TYPES = list(CFG.expected_edit_types)
print("Confirmed edit types:", CONFIRMED_EDIT_TYPES)

# ── Field name constants (used by download + audit code) ──────────────────
COL_INPUT_IMAGES  = "input_images"   # list[str] — image paths
COL_OUTPUT_IMAGES = "output_images"  # list[str] — image paths
COL_PROMPT        = "prompt"         # str

EDIT_TYPE_SOURCE  = "filename_prefix"
# Rule: first "_"-separated token of parquet filename stem before "part{N}"
# Implementation: infer_edit_type_from_filename() in §3.1 utilities.

# ── Sample-ID extraction ───────────────────────────────────────────────────
# path = "results_background_laion_part0/00121_00035_000359496/original.png"
# sample_id = path.split("/")[1]  → "00121_00035_000359496"
SAMPLE_ID_PATH_COMPONENT = 1   # index into "/" split of image path

# ── Mask dataset (sysuyy/ImgEdit_recap_mask) — CONFIRMED from §2.3 ────────
# File format : tar archives (NOT parquet)
#   jsons/part0.tar … jsons/part7.tar  — individual JSON per sample (~47k/tar)
#   laion-aes/00000.tar … — image archives (not needed for mask loading)
#
# Top-level JSON fields (confirmed §2.3):
#   'path'        : str  — image path, e.g. '00027/00038/000386475.jpg'
#                          This is the JOIN KEY after normalization (see below).
#   'cap'         : list[str]   — image caption(s)
#   'resolution'  : dict        — {'height': int, 'width': int}
#   'aes'         : float       — LAION aesthetic score
#   'border'      : list[int]   — border crop info
#   'tags'        : dict        — {'background': ..., 'object': ..., 'summary': ...}
#   'segmentation': dict        — {'background': list, 'object': list, 'box_format': str}
#   'bg_count'    : dict        — per-class background object counts
#   'obj_count'   : dict        — per-class foreground object counts
#
# segmentation field structure:
#   segmentation['object']     : list of per-object annotation dicts
#   segmentation['background'] : list of per-background annotation dicts
#   segmentation['box_format'] : bbox coordinate format (e.g. 'xyxy' or 'xywh')
#   Each annotation dict has: 'class_name', 'bbox' (list[4 float])
#   Inner RLE mask field name: TBD — see §2.3 step 9 output, then set
#                              MASK_INNER_RLE_FIELD in §3.1.
#
# Join key normalization:
#   mask 'path' = '00027/00038/000386475.jpg'
#   → sample_id = '00027_00038_000386475'   (replace '/' with '_', strip ext)
#   This matches extract_sample_id() output for laion-aes ImgEdit samples.
#   Action-partition video-frame samples will produce no-match → empty annotations.
MASK_JOIN_KEY       = "path"   # confirmed — normalize via normalize_mask_path_to_sample_id()

print("\nField constants:")
print(f"  COL_INPUT_IMAGES  = {COL_INPUT_IMAGES!r}")
print(f"  COL_OUTPUT_IMAGES = {COL_OUTPUT_IMAGES!r}")
print(f"  COL_PROMPT        = {COL_PROMPT!r}")
print(f"  EDIT_TYPE_SOURCE  = {EDIT_TYPE_SOURCE!r}")
print(f"  SAMPLE_ID_PATH_COMPONENT = {SAMPLE_ID_PATH_COMPONENT}")
print(f"  MASK_JOIN_KEY     = {MASK_JOIN_KEY!r}")
print(f"  mask_dataset_id   = {CFG.mask_dataset_id!r}")
print()
print("Proceed to §2.3 to confirm mask dataset schema before writing download code.")


## §2.3 — Mask Dataset Schema Inspection (`sysuyy/ImgEdit_recap_mask`)
*Read-only. Run before §4 download to confirm join key and RLE field structure.*

**Confirmed format (from §2.1 listing):** The mask dataset contains **no parquet files**.
Annotations are stored as JSON/JSONL files inside tar archives:
- `jsons/part0.tar` … `jsons/part7.tar` — one JSON (or JSONL) per sample, containing bbox + RLE mask data
- `laion-aes/00000.tar` … — image archives (not needed for mask loading)

This cell downloads `jsons/part0.tar`, extracts and prints the first few entries,
identifies the join key field and mask/bbox fields, then prints a summary for §2.2 / §3.1 updates.


In [ ]:
# ── §2.3  Inspect sysuyy/ImgEdit_recap_mask schema ────────────────────────
# Dataset format: tar archives — no parquet files.
# jsons/partN.tar  → JSON or JSONL annotation files (bbox + RLE per sample)
# laion-aes/*.tar  → raw images (not loaded here)
# This cell is READ-ONLY. Re-run freely.

import tarfile, io, json as _json
from huggingface_hub import HfApi, hf_hub_download
from pprint import pformat

HF_MASK_ID = CFG.mask_dataset_id   # "sysuyy/ImgEdit_recap_mask"
INSPECT_N  = 3                      # number of JSON entries to inspect

api = HfApi()

# ── 1. List all mask dataset files ────────────────────────────────────────
print(f"{'='*60}")
print(f"Listing files in mask dataset: {HF_MASK_ID}")
print(f"{'='*60}")
try:
    mask_files  = sorted(api.list_repo_files(HF_MASK_ID, repo_type="dataset"))
    mask_parqs  = [f for f in mask_files if f.endswith(".parquet")]
    mask_tars   = [f for f in mask_files if f.endswith(".tar")]
    json_tars   = sorted(f for f in mask_tars if f.startswith("jsons/"))
    image_tars  = sorted(f for f in mask_tars if not f.startswith("jsons/"))

    print(f"Found {len(mask_files)} files total:")
    for f in mask_files[:20]:
        print(f"  {f}")
    if len(mask_files) > 20:
        print(f"  ... and {len(mask_files)-20} more")
    print(f"\nParquet files:  {len(mask_parqs)}  (expected 0 — confirmed no parquet)")
    print(f"JSON tar files: {len(json_tars)}")
    print(f"Image tar files:{len(image_tars)}")

    # Fail-fast if structure changes unexpectedly
    if mask_parqs:
        print("WARNING: unexpected parquet files found — update downstream code.")
    if not json_tars:
        raise RuntimeError(
            "No jsons/*.tar files found — mask dataset structure may have changed. "
            "Inspect manually before proceeding."
        )
except Exception as e:
    print(f"ERROR listing mask dataset: {e}")
    raise

# ── 2. Download first JSON tar and peek inside ─────────────────────────────
inspect_tar = json_tars[0]
print(f"\n{'='*60}")
print(f"Downloading: {inspect_tar}")
print(f"{'='*60}")
local_tar_path = hf_hub_download(
    repo_id=HF_MASK_ID, repo_type="dataset", filename=inspect_tar
)
print(f"Saved to: {local_tar_path}")

# ── 3. Inspect tar member list ────────────────────────────────────────────
with tarfile.open(local_tar_path, "r:*") as tf:
    members = tf.getmembers()
    print(f"\nTotal members in tar: {len(members):,}")
    print("First 10 entries:")
    for m in members[:10]:
        print(f"  {m.name:<60s}  {m.size:>8,} bytes")
    if len(members) > 10:
        print(f"  ... and {len(members)-10} more")

    # ── 4. Classify member format ─────────────────────────────────────────
    json_members  = [m for m in members if m.name.endswith(".json")]
    jsonl_members = [m for m in members if m.name.endswith(".jsonl")]

    print(f"\n  .json files:  {len(json_members)}")
    print(f"  .jsonl files: {len(jsonl_members)}")

    # ── 5. Extract sample entries ─────────────────────────────────────────
    sample_entries = []   # list of (label, dict_or_list)

    if json_members:
        # Individual JSON per annotation
        print(f"\nExtracting first {INSPECT_N} JSON files...")
        for m in json_members[:INSPECT_N]:
            fobj = tf.extractfile(m)
            if fobj is None:
                continue
            data = _json.loads(fobj.read().decode("utf-8"))
            sample_entries.append((m.name, data))

    elif jsonl_members:
        # Single JSONL with one entry per line
        print(f"\nExtracting first {INSPECT_N} lines from {jsonl_members[0].name}...")
        fobj = tf.extractfile(jsonl_members[0])
        if fobj:
            for _ in range(INSPECT_N):
                line = fobj.readline()
                if not line.strip():
                    break
                sample_entries.append((jsonl_members[0].name, _json.loads(line)))
    else:
        # Unknown structure — show raw names and raise
        print("WARNING: no .json or .jsonl files found inside tar.")
        print("Raw member names:")
        for m in members[:20]:
            print(f"  {m.name}")
        raise RuntimeError(
            "Unexpected tar structure — cannot determine JSON format. "
            "Inspect manually before continuing."
        )

# ── 6. Print schema for each sample entry ────────────────────────────────
print(f"\n{'='*60}")
print("RAW SAMPLE ENTRIES — full field listing:")
print(f"{'='*60}")

def _describe_value(val, max_len=120):
    """Return a short human-readable description of a value."""
    if isinstance(val, (bytes, bytearray)):
        return f"bytes len={len(val)}"
    if isinstance(val, dict):
        return f"dict  keys={list(val.keys())}"
    if isinstance(val, list):
        inner = str(val[0])[:60] if val else "empty"
        return f"list[{len(val)}]  first={inner!r}"
    s = str(val)
    return f"{type(val).__name__} = {s[:max_len]!r}{'...' if len(s)>max_len else ''}"

for label, data in sample_entries:
    print(f"\n  Source: {label}")
    if isinstance(data, dict):
        for k, v in data.items():
            print(f"    {k!r:<30s} {_describe_value(v)}")
    elif isinstance(data, list):
        print(f"  [list of {len(data)} items]")
        if data and isinstance(data[0], dict):
            print(f"  First item:")
            for k, v in data[0].items():
                print(f"    {k!r:<30s} {_describe_value(v)}")
    else:
        print(f"  [unexpected type {type(data).__name__}]: {str(data)[:200]!r}")

# ── 7. Candidate field analysis ───────────────────────────────────────────
print(f"\n{'='*60}")
print("FIELD ANALYSIS — join key + mask/bbox candidates:")
print(f"{'='*60}")

first_data = sample_entries[0][1] if sample_entries else {}
fields = first_data.items() if isinstance(first_data, dict) else (
    first_data[0].items() if isinstance(first_data, list) and first_data else []
)
all_keys = []
for k, v in fields:
    all_keys.append(k)
    kl = k.lower()
    vs = str(v)
    if any(kw in kl for kw in ["id", "name", "file", "image", "key", "path"]):
        print(f"  JOIN KEY CANDIDATE :  {k!r:<28s} = {vs[:80]!r}")
    if any(kw in kl for kw in ["mask", "rle", "seg", "count", "ann", "region"]):
        print(f"  MASK/RLE CANDIDATE :  {k!r:<28s} = {vs[:80]!r}")
    if any(kw in kl for kw in ["bbox", "box", "rect", "coord", "xyxy", "xywh"]):
        print(f"  BBOX CANDIDATE     :  {k!r:<28s} = {vs[:80]!r}")

print(f"\nAll top-level keys: {all_keys}")

# ── 8. Summary of confirmed top-level schema ─────────────────────────────
print(f"\n{'='*60}")
print("CONFIRMED: mask dataset uses tar+JSON (not parquet).")
print(f"  JSON tars:  {json_tars}")
print()
print("Confirmed top-level fields:")
print("  Join key:      'path'         — normalize with normalize_mask_path_to_sample_id()")
print("  Annotation:    'segmentation' — nested dict with 'background', 'object', 'box_format'")
print("  Image size:    'resolution'   — {'height': int, 'width': int}")
print("  Object counts: 'obj_count'    — {class_name: count}")
print()
print("ACTION: Run step 9 below to reveal inner annotation (RLE/mask) field names.")
print(f"{'='*60}")

# ── 9. Deep drill — segmentation inner structure ──────────────────────────
# Why: the 'segmentation' field is a nested dict; we need to know the
# per-annotation field names (especially the RLE mask key) before writing
# _index_mask_record() in §3.1.  This step prints each field of a real
# annotation object so MASK_INNER_RLE_FIELD can be set with certainty.

print(f"\n{'='*60}")
print("STEP 9 — segmentation inner structure (per-object annotation):")
print(f"{'='*60}")

if sample_entries:
    first_doc = sample_entries[0][1] if isinstance(sample_entries[0], tuple) else sample_entries[0]
    seg = first_doc.get('segmentation', {}) if isinstance(first_doc, dict) else {}

    print(f"\nTop-level segmentation keys: {list(seg.keys())}")
    print(f"box_format = {seg.get('box_format')!r}")

    for region_key in ['object', 'background']:
        region_list = seg.get(region_key, [])
        print(f"\nsegmentation[{region_key!r}]: {len(region_list)} annotation(s)")
        if region_list:
            ann0 = region_list[0]
            if isinstance(ann0, dict):
                print(f"  First annotation keys: {list(ann0.keys())}")
                for k, v in ann0.items():
                    if isinstance(v, (bytes, bytearray)):
                        print(f"    {k!r:<30s} bytes len={len(v)}")
                    elif isinstance(v, dict):
                        print(f"    {k!r:<30s} dict  keys={list(v.keys())}")
                    elif isinstance(v, list):
                        inner = str(v[0])[:60] if v else "empty"
                        print(f"    {k!r:<30s} list[{len(v)}]  first={inner!r}")
                    elif isinstance(v, str) and len(v) > 100:
                        print(f"    {k!r:<30s} str (len={len(v)}) = {v[:80]!r}...")
                    else:
                        print(f"    {k!r:<30s} {type(v).__name__} = {str(v)[:100]!r}")
            else:
                print(f"  [Non-dict item: {type(ann0).__name__}] = {str(ann0)[:200]!r}")
        else:
            print("  (empty list in this sample)")
else:
    print("WARNING: no sample entries loaded — re-run from step 2.")

print(f"\n{'='*60}")
print("AFTER reviewing step 9 output:")
print("  1. Set MASK_INNER_RLE_FIELD in §3.1 to the confirmed RLE key")
print("     (likely 'mask', 'segmentation', or a COCO-RLE dict field)")
print("  2. Verify box_format matches spec (expected: 'xyxy', absolute pixels)")
print("  3. §2.3 is now complete — proceed to §3.1 and §4.1.")
print(f"{'='*60}")


## §2.4 — Confirmed Mask Schema Contract
*Run after §2.3 to propagate confirmed field names to §3.1. All values confirmed from §2.3 step 9 output.*

**Confirmed from §2.3 step 9:**
- `MASK_INNER_RLE_FIELD = 'mask'` — the per-annotation field containing the compressed RLE string
- `MASK_BOX_FORMAT = 'xyxy'` — bbox coords are absolute pixels, x1/y1/x2/y2 order
- RLE is stored as a **plain string** (COCO compressed format), NOT a dict — must be paired with `resolution` to build a decodable COCO RLE dict
- Additional per-annotation fields available: `score` (list[1] float), `clip_score` (float), `aes_score` (float) — useful for quality filtering

Run this cell once to confirm all constants, then proceed to §3.1 and §4.1.


In [ ]:
# ── §2.4  Confirmed mask schema constants — CONFIRMED from §2.3 step 9 ──
# These are the ONLY place mask field names should be defined.
# §3.1 imports them; do NOT duplicate these strings elsewhere.
#
# ── CONFIRMED from §2.3 step 9 output ────────────────────────────────────
#
# MASK_INNER_RLE_FIELD: the field name inside each segmentation.object[i] dict
#   that contains the mask. Confirmed: 'mask' — a compressed RLE string
#   (COCO format, e.g. 'kgYi13^_1c0Cd0...'). NOT a JSON dict — must be
#   paired with resolution to form {'counts': ..., 'size': [H, W]}.
MASK_INNER_RLE_FIELD = "mask"    # ← CONFIRMED from §2.3 step 9

# MASK_BOX_FORMAT: value of segmentation['box_format'] from step 9.
#   Confirmed: 'xyxy' — absolute pixels, matching spec §3.2.
#   No conversion needed in bbox_to_xyxy().
MASK_BOX_FORMAT = "xyxy"          # ← CONFIRMED from §2.3 step 9

# ── Already confirmed from §2.3 steps 1-8 ────────────────────────────────
MASK_JOIN_FIELD        = "path"          # top-level field used as join key
MASK_SEG_FIELD         = "segmentation" # top-level field containing all annotations
MASK_RESOLUTION_FIELD  = "resolution"   # top-level field with {'height', 'width'}
MASK_OBJ_COUNT_FIELD   = "obj_count"    # top-level field with per-class object counts

# Inside segmentation dict:
MASK_OBJ_LIST_KEY      = "object"       # list of foreground annotation dicts
MASK_BG_LIST_KEY       = "background"   # list of background annotation dicts
MASK_BOX_FORMAT_KEY    = "box_format"   # string field declaring bbox format

# Inside each annotation dict:
MASK_CLASS_NAME_KEY    = "class_name"   # str — object class label
MASK_BBOX_KEY          = "bbox"         # list[4 float] — coords in MASK_BOX_FORMAT
# MASK_INNER_RLE_FIELD  = 'mask'        — compressed RLE string (confirmed above)

# Optional quality fields (confirmed present per annotation):
MASK_SCORE_KEY         = "score"        # list[1 float] — SAM2 confidence score
MASK_CLIP_SCORE_KEY    = "clip_score"   # float — CLIP relevance score
MASK_AES_SCORE_KEY     = "aes_score"    # float — aesthetic score

# ── Validation ────────────────────────────────────────────────────────────
print("§2.4 mask schema constants (all confirmed from §2.3 step 9):")
print(f"  MASK_JOIN_FIELD        = {MASK_JOIN_FIELD!r}")
print(f"  MASK_SEG_FIELD         = {MASK_SEG_FIELD!r}")
print(f"  MASK_RESOLUTION_FIELD  = {MASK_RESOLUTION_FIELD!r}")
print(f"  MASK_OBJ_COUNT_FIELD   = {MASK_OBJ_COUNT_FIELD!r}")
print(f"  MASK_OBJ_LIST_KEY      = {MASK_OBJ_LIST_KEY!r}")
print(f"  MASK_BG_LIST_KEY       = {MASK_BG_LIST_KEY!r}")
print(f"  MASK_BOX_FORMAT_KEY    = {MASK_BOX_FORMAT_KEY!r}")
print(f"  MASK_CLASS_NAME_KEY    = {MASK_CLASS_NAME_KEY!r}")
print(f"  MASK_BBOX_KEY          = {MASK_BBOX_KEY!r}")
print(f"  MASK_INNER_RLE_FIELD   = {MASK_INNER_RLE_FIELD!r}  ✓ CONFIRMED")
print(f"  MASK_BOX_FORMAT        = {MASK_BOX_FORMAT!r}  ✓ CONFIRMED")
print(f"  MASK_SCORE_KEY         = {MASK_SCORE_KEY!r}")
print(f"  MASK_CLIP_SCORE_KEY    = {MASK_CLIP_SCORE_KEY!r}")
print(f"  MASK_AES_SCORE_KEY     = {MASK_AES_SCORE_KEY!r}")
print()
assert MASK_INNER_RLE_FIELD is not None, "MASK_INNER_RLE_FIELD must not be None"
assert MASK_BOX_FORMAT in ("xyxy", "xywh"), f"Unexpected box_format: {MASK_BOX_FORMAT!r}"
print("✓ All mask schema constants confirmed. Ready to run §3.1 and §4.1.")


## §2.5 — Main Dataset Repo File Structure Inspection
*Run this once before §4.1 to discover image tar file names. Images in `sysuyy/ImgEdit` are NOT accessible as individual files — they live in tar archives.*

**Confirmed from §6.1 smoke test:** HfFileSystem and hf_hub_download per-file fetch both fail for all image paths. The repo stores images in tar archives.

This cell lists all files in `sysuyy/ImgEdit`, identifies tar archives, and maps image path prefixes → tar filenames so §3.1 `build_image_tar_names()` can fetch images correctly.


In [ ]:
# ── §2.5  Inspect main ImgEdit repo — discover split tar structure ──────────
# Updated §2.6: images are stored as .tar.split.NNN files (NOT .tar files).
# The only .tar in the main repo is Benchmark.tar (irrelevant).
# This cell lists all .tar.split.* files, groups them by stem, and maps
# image path prefixes (from parquet input_images) to tar stem groups.
#
# Confirmed from §2.5 v2.5 output:
#   - Total tar files: 1 (Benchmark.tar — irrelevant)
#   - Split tar files: ~290+ files in Singleturn/ and Multiturn/
#   - Path prefix "results_compose_part0" and "results_extract_ref_part1" → NO TAR

import re as _re
from huggingface_hub import HfApi
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
import numpy as np

api     = HfApi()
HF_MAIN = CFG.hf_dataset_id

print(f"{'='*60}")
print(f"Listing ALL files in: {HF_MAIN}")
print(f"{'='*60}")

all_main_files = sorted(api.list_repo_files(HF_MAIN, repo_type="dataset"))
parquet_files  = [f for f in all_main_files if f.endswith(".parquet")]
plain_tars     = [f for f in all_main_files if f.endswith(".tar")]
split_tars     = [f for f in all_main_files
                  if _re.search(r'\.tar\.split\.\d+$', f)]
other_files    = [f for f in all_main_files
                  if not f.endswith(".parquet")
                  and not f.endswith(".tar")
                  and not _re.search(r'\.tar\.split\.\d+$', f)]

print(f"Total files         : {len(all_main_files)}")
print(f"Parquet files       : {len(parquet_files)}")
print(f"Plain .tar files    : {len(plain_tars)}  (only Benchmark.tar — irrelevant)")
print(f"Split .tar.split.*  : {len(split_tars)}")
print(f"Other               : {len(other_files)}")

# ── Group split tars by stem ────────────────────────────────────────────────
# Stem = basename without .tar.split.NNN suffix
# e.g. "Singleturn/results_background_laion_part5.tar.split.000" → stem "results_background_laion_part5"
stem_groups = {}   # stem → sorted list of HF paths
for hf_path in split_tars:
    basename = hf_path.split("/")[-1]
    m = _re.match(r"^(.+)\.tar\.split\.\d+$", basename)
    if m:
        stem = m.group(1)
        stem_groups.setdefault(stem, []).append(hf_path)

print(f"\nUnique tar stems    : {len(stem_groups)}")
print("\nAll stems (sorted):")
for stem in sorted(stem_groups):
    parts = stem_groups[stem]
    print(f"  {stem:<55s}  [{len(parts)} part(s)]")

# ── Build path-prefix → stem mapping using 4-rule fuzzy matching ─────────────
# Rules (in priority):
#  1. Exact: stem == prefix
#  2. Starts-with: stem starts with prefix + "_"
#  3. results_ + exact: stem == "results_" + prefix
#  4. results_ + starts-with: stem starts with "results_" + prefix
def find_stems_for_prefix(prefix, all_stems):
    exact, sw, r_exact, r_sw = [], [], [], []
    for s in all_stems:
        if s == prefix:                          exact.append(s)
        elif s.startswith(prefix + "_"):         sw.append(s)
        elif s == "results_" + prefix:           r_exact.append(s)
        elif s.startswith("results_" + prefix):  r_sw.append(s)
    return exact + sw + r_exact + r_sw

# ── Cross-check: sample parquets to observe real path prefixes ───────────────
print(f"\n{'='*60}")
print("Cross-checking parquet path prefixes against tar stems:")
print(f"{'='*60}")

def _safe_first_path(val):
    if isinstance(val, (list, np.ndarray)) and len(val) > 0:
        return str(val[0])
    return str(val)

prefix_to_stems = {}
for pq_file in [f for f in parquet_files if "part0" in f][:8]:
    try:
        local_pq = hf_hub_download(repo_id=HF_MAIN, repo_type="dataset", filename=pq_file)
        table = __import__("pyarrow.parquet", fromlist=["read_table"]).read_table(local_pq)
        df = table.to_pandas()
        data_col = "input_images" if "input_images" in df.columns else None
        if data_col is None:
            data_cols = [c for c in df.columns if not c.startswith("__")]
            if data_cols:
                sample = df[data_cols[0]].dropna().iloc[0] if len(df) > 0 else None
                if isinstance(sample, list) and sample and isinstance(sample[0], dict):
                    df = __import__("pandas").DataFrame([r for cell in df[data_cols[0]] for r in (cell if isinstance(cell, list) else [cell])])
                    if "input_images" in df.columns:
                        data_col = "input_images"
        if data_col:
            for val in df[data_col].head(3):
                p = _safe_first_path(val)
                prefix = p.split("/")[0]
                if prefix not in prefix_to_stems:
                    stems = find_stems_for_prefix(prefix, stem_groups.keys())
                    prefix_to_stems[prefix] = stems
    except Exception as e:
        print(f"  WARNING sampling {pq_file}: {e}")

print("\nPath prefix → matching tar stem(s):")
all_ok = True
for prefix, stems in sorted(prefix_to_stems.items()):
    status = "✓" if stems else "✗ NO TAR"
    print(f"  {status}  {prefix!r:<45s} → {stems}")
    if not stems:
        all_ok = False

missing_prefixes = [p for p, s in prefix_to_stems.items() if not s]
if missing_prefixes:
    print(f"\nWARNING: {len(missing_prefixes)} prefix(es) have no matching tar:")
    for p in missing_prefixes:
        print(f"  {p!r}  — samples with this prefix will be SKIPPED in download")
    print("These are likely 'compose' (no data released) and 'reference' (extract_ref).")
else:
    print("\n✓ All observed prefixes have matching tars.")

print(f"\n{'='*60}")
print("§2.5 complete (v2.6 — split tar indexing).")
print(f"  {len(stem_groups)} unique tar stems covering ~{sum(len(v) for v in stem_groups.values())} split files")
print("  build_image_tar_names() in §3.1 will use this split-tar mapping.")
print(f"{'='*60}")


## §3 — Shared Utilities

In [ ]:
# ── §3.1  Shared utilities used across all phases ─────────────────────────

import io, json, logging, re, tarfile
import numpy as np
from pathlib import Path
from typing import Any, Optional
from PIL import Image

# ── Logging setup ─────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler()],
)
log = logging.getLogger("pipeline")

# ── Image split-tar index — populated by build_image_tar_names() ────────────
# Confirmed §2.5 v2.6: images are in *.tar.split.NNN split archives.
# The only plain .tar is Benchmark.tar (unrelated).
#
# _IMAGE_TAR_SPLITS: stem → sorted list of HF split filenames
#   e.g. "results_background_laion_part5" → ["Singleturn/results_background_laion_part5.tar.split.000",
#                                             "Singleturn/results_background_laion_part5.tar.split.001"]
# _IMAGE_TAR_LOCAL : stem → local path of reassembled .tar file (Drive-cached)
#
# Fetching strategy:
#   1. Find matching stem(s) via 4-rule fuzzy matching on path prefix
#   2. Download all .tar.split.* parts for matching stems
#   3. Concatenate parts → single .tar on Drive  4. Extract image from reassembled tar
_IMAGE_TAR_SPLITS: dict = {}   # stem → sorted list of HF split-file paths
_IMAGE_TAR_LOCAL:  dict = {}   # stem → local reassembled .tar path

# ── RLE decode ────────────────────────────────────────────────────────────
import pycocotools.mask as mask_utils

def decode_rle_mask(rle: dict) -> Optional[np.ndarray]:
    """Decode a COCO-format RLE dict to a binary uint8 mask (H×W).

    Handles both compressed RLE (counts as str) and uncompressed (counts as list).
    Returns None on any failure — callers must treat None as invalid_rle.
    """
    if rle is None:
        return None
    try:
        rle_copy = dict(rle)
        if isinstance(rle_copy.get("counts"), str):
            rle_copy["counts"] = rle_copy["counts"].encode("utf-8")
        mask = mask_utils.decode(rle_copy)
        assert mask.ndim == 2, f"Expected 2-D mask, got {mask.shape}"
        return mask
    except Exception as e:
        log.debug(f"RLE decode failed: {e}")
        return None


# ── Edit-type inference ────────────────────────────────────────────────────
KNOWN_EDIT_TYPES = set(CFG.expected_edit_types)

def infer_edit_type_from_filename(fname: str) -> str:
    """Extract edit type from parquet filename prefix.

    Pattern: [dir/]{edit_type}[_{qualifier}]_part{N}.parquet
    Examples:
        Parquet/background_part0.parquet   → "background"
        Parquet/adjust_canny_part0.parquet → "adjust"
        Parquet/content_memory_part2.parquet → "content"
    Why: edit_type is not a parquet column — it lives only in the filename.
    """
    stem = fname.split("/")[-1].replace(".parquet", "")
    tokens = stem.split("_")
    accumulated = ""
    for tok in tokens:
        if tok.startswith("part") and tok[4:].isdigit():
            break
        accumulated = (accumulated + "_" + tok).lstrip("_")
        if accumulated in KNOWN_EDIT_TYPES:
            return accumulated
    return tokens[0] if tokens else "unknown"


# ── Sample-ID extraction from image path ──────────────────────────────────
def extract_sample_id(image_path: str) -> str:
    """Extract sample_id from an ImgEdit image path.

    Handles two confirmed path patterns:
      1. Laion-AES style (background, add, remove, etc.):
           results_{edit_type}_laion_part{N}/{sample_id}/{filename}
           e.g.  results_background_laion_part0/00121_00035_000359496/original.png
           → "00121_00035_000359496"
      2. Video-frame style (action partition — flat filename):
           hOdCsLRNAyw_segment_21_frame_0.jpg
           → "hOdCsLRNAyw_segment_21_frame_0"

    The returned id is the join key for sysuyy/ImgEdit_recap_mask.
    Action-partition samples will produce no-match in the mask index
    (mask dataset is laion-AES only) — _load_mask_for_sample() returns
    empty annotations for them, which is expected and correct.
    """
    parts = image_path.replace("\\", "/").split("/")
    if len(parts) >= 2:
        return parts[1]                      # laion-AES pattern (subdirectory = sample_id)
    # Flat filename (video frames, action partition) — strip extension
    return parts[0].rsplit(".", 1)[0]


def normalize_mask_path_to_sample_id(mask_path: str) -> str:
    """Normalize a mask dataset 'path' field to match extract_sample_id() output.

    Mask path: '00027/00038/000386475.jpg'  (confirmed format §2.3)
    Output:    '00027_00038_000386475'       (matches sample_id from main dataset)

    Normalization: strip extension, replace '/' separators with '_'.
    This produces identical strings to the laion-AES extract_sample_id() path.

    Why a separate function: the two datasets store the same logical ID in
    different string formats; centralising the conversion prevents join misses.
    """
    stem = mask_path.rsplit(".", 1)[0]   # strip extension
    return stem.replace("/", "_")        # '00027/00038/000386475' → '00027_00038_000386475'


# ── Bbox utilities ────────────────────────────────────────────────────────
def bbox_area_absolute(bbox_xyxy: list) -> float:
    x1, y1, x2, y2 = bbox_xyxy
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def bbox_to_relative(bbox_xyxy: list, W: int, H: int) -> list:
    """Convert absolute xyxy to Qwen [0,1000] relative scale."""
    x1, y1, x2, y2 = bbox_xyxy
    return [round(x1/W*1000), round(y1/H*1000), round(x2/W*1000), round(y2/H*1000)]

def bbox_relative_to_absolute(bbox_rel: list, W: int, H: int) -> list:
    x1, y1, x2, y2 = bbox_rel
    return [x1/1000*W, y1/1000*H, x2/1000*W, y2/1000*H]


# ── Bbox routing (single authoritative implementation) ────────────────────
GLOBAL_ADJUST_KEYWORDS = set(CFG.global_adjust_keywords)

def route_bbox(edit_type: str, edit_description: str,
               annotations: list, img_W: int, img_H: int) -> list:
    """Apply edit_type-aware bbox routing per spec §3.2.

    Returns bbox in [0,1000] relative scale.
    Called identically from build_vlm_dataset() and audit_dataset()
    — there is exactly one routing implementation.
    """
    FULL_IMAGE = [0, 0, 1000, 1000]
    et = (edit_type or "").strip().lower()

    if et in {"style", "background"}:
        return FULL_IMAGE

    if et == "adjust":
        desc_lower = (edit_description or "").lower()
        if any(kw in desc_lower for kw in GLOBAL_ADJUST_KEYWORDS):
            return FULL_IMAGE
        # local adjust — fall through to largest-object logic

    best_bbox, best_area = None, 0.0
    for ann in (annotations or []):
        raw_bbox = ann.get("bbox")
        if raw_bbox is None or len(raw_bbox) != 4:
            continue
        area = bbox_area_absolute(raw_bbox)
        if area > best_area:
            best_area = area
            best_bbox = raw_bbox

    if best_bbox is None or best_area == 0.0:
        return FULL_IMAGE
    return bbox_to_relative(best_bbox, img_W, img_H)


# ── Split-tar streaming architecture (v2.9) ──────────────────────────────────
# Confirmed §2.6: each split file ≈ 42.9 GB; reassembling ALL splits to disk
# is infeasible (357 GB per tar > 235 GB Drive).
#
# Strategy: SplitStreamReader downloads one split to Colab /tmp at a time,
# feeds it to tarfile in streaming mode (r|), extracts needed members, then
# deletes the split before downloading the next.
#
# Peak disk:
#   /tmp  (Colab local): 1 split file × 42.9 GB  ✓ (Colab local disk ~100 GB)
#   Drive                : extracted JPEGs only   ✓ (~0.1 MB each)
# Never assembles full tar. Zero Drive usage for archive files.

import re as _re
import shutil

def build_image_tar_names(hf_dataset_id: str = None) -> dict:
    """Index all .tar.split.* archives by stem.

    Confirmed §2.5 v2.6: images live in split archives (*.tar.split.NNN).
    Groups files by stem, e.g. all results_background_laion_part5.tar.split.*
    → stem "results_background_laion_part5".

    Does NOT download anything. Safe to call multiple times (no-op if done).
    Returns _IMAGE_TAR_SPLITS: {stem: [sorted hf_split_path, ...]}
    """
    global _IMAGE_TAR_SPLITS
    if _IMAGE_TAR_SPLITS:
        return _IMAGE_TAR_SPLITS
    hf_id = hf_dataset_id or CFG.hf_dataset_id
    api = HfApi()
    all_files = list(api.list_repo_files(hf_id, repo_type="dataset"))
    for hf_path in all_files:
        basename = hf_path.split("/")[-1]
        m = _re.match(r"^(.+)\.tar\.split\.\d+$", basename)
        if m:
            stem = m.group(1)
            _IMAGE_TAR_SPLITS.setdefault(stem, []).append(hf_path)
    for stem in _IMAGE_TAR_SPLITS:
        _IMAGE_TAR_SPLITS[stem] = sorted(_IMAGE_TAR_SPLITS[stem])
    log.info(f"Split-tar index: {len(_IMAGE_TAR_SPLITS)} stems from {hf_id}")
    return _IMAGE_TAR_SPLITS


def _find_tar_stems_for_prefix(prefix: str) -> list:
    """Return tar stems likely containing images for the given path prefix.

    4-rule priority (confirmed §2.5 v2.6):
      1. Exact: stem == prefix
      2. Starts-with: stem starts with prefix + "_"
      3. results_+exact: stem == "results_" + prefix
      4. results_+starts-with: stem starts with "results_" + prefix

    Returns [] for known-unavailable prefixes (results_compose_part0,
    results_extract_ref_part1).
    """
    if not _IMAGE_TAR_SPLITS:
        return []
    exact, sw, r_exact, r_sw = [], [], [], []
    rp = "results_" + prefix
    for s in _IMAGE_TAR_SPLITS:
        if s == prefix:                        exact.append(s)
        elif s.startswith(prefix + "_"):       sw.append(s)
        elif s == rp:                          r_exact.append(s)
        elif s.startswith(rp):                 r_sw.append(s)
    return exact + sw + r_exact + r_sw


class SplitStreamReader(io.RawIOBase):
    """Sequential multi-part reader for .tar.split.NNN archives.

    Downloads each split to Colab /tmp ONE AT A TIME, streams it to tarfile,
    then deletes it before fetching the next split.

    Peak /tmp disk : 1 split file at a time (~42.9 GB).
    Peak Drive disk: extracted JPEGs only (~0.1 MB each).
    Never assembles full tar — no Drive storage required for archives.

    Use with: tarfile.open(fileobj=SplitStreamReader(...), mode="r|")
    The "r|" (streaming) mode is required — the reader is NOT seekable.

    Tarfile streaming notes:
    - Iterates members sequentially; cannot jump to a specific member.
    - Caller MUST read extractfile() result immediately before next iteration.
    - Caller should break early once all needed members are found (efficiency).
    """
    CHUNK = 1 << 20  # 1 MB read chunks

    def __init__(self, split_hf_files: list, hf_dataset_id: str,
                 tmp_dir: Optional[Path] = None):
        super().__init__()
        self.split_hf_files = sorted(split_hf_files)
        self.hf_dataset_id  = hf_dataset_id
        self.tmp_dir        = Path(tmp_dir or "/tmp/split_stream_tmp")
        self.tmp_dir.mkdir(parents=True, exist_ok=True)
        self._split_idx     = 0      # which split file we're currently on
        self._current_local = None   # local path of current split in /tmp
        self._current_fh    = None   # open file handle
        self._advance()              # download + open split[0]

    def _advance(self):
        """Download next split to /tmp; delete previous split first."""
        if self._current_fh:
            self._current_fh.close()
            self._current_fh = None
        if self._current_local and Path(self._current_local).exists():
            Path(self._current_local).unlink()
            log.debug(f"Deleted split: {Path(self._current_local).name}")
        self._current_local = None

        if self._split_idx >= len(self.split_hf_files):
            return   # signals EOF — readinto will return 0

        hf_fname = self.split_hf_files[self._split_idx]
        log.info(
            f"    Downloading split "
            f"{self._split_idx + 1}/{len(self.split_hf_files)}: "
            f"{hf_fname.split('/')[-1]} (~42.9 GB) ..."
        )
        # Download to /tmp (Colab local disk), NOT Drive.
        # local_dir_use_symlinks=False: write actual bytes, no HF-cache symlink.
        local = hf_hub_download(
            repo_id=self.hf_dataset_id, repo_type="dataset", filename=hf_fname,
            local_dir=str(self.tmp_dir), local_dir_use_symlinks=False,
        )
        self._current_local = local
        self._current_fh    = open(local, "rb")
        self._split_idx    += 1
        log.info(f"    Ready: {Path(local).name}")

    def readinto(self, b: bytearray) -> int:
        if self.closed or self._current_fh is None:
            return 0
        while True:
            n = self._current_fh.readinto(b)
            if n:
                return n
            # Split exhausted — try next split
            self._advance()
            if self._current_fh is None:
                return 0   # all splits consumed

    def readable(self) -> bool:
        return True

    def close(self):
        if not self.closed:
            if self._current_fh:
                try: self._current_fh.close()
                except: pass
            if self._current_local and Path(self._current_local).exists():
                try: Path(self._current_local).unlink()
                except: pass
        super().close()


def extract_images_streaming(
    stem: str,
    needed: dict,
    hf_dataset_id: str,
    images_dir: Path,
    tmp_dir: Optional[Path] = None,
) -> dict:
    """Extract needed images from a split-tar archive using streaming.

    Scans the tar archive sequentially (streaming mode). For each member whose
    path matches a needed image, decodes it to a PIL Image and saves as JPEG.
    Stops scanning early once all needed images are found.

    Args:
        stem:          Tar stem, e.g. "results_background_laion_part5".
        needed:        dict mapping member_path_variants → (sid, role)
                       where role is "src" or "tgt".
                       member_path_variants is a frozenset of paths to try
                       (full path, no-prefix, basename).
        hf_dataset_id: HuggingFace dataset repo ID.
        images_dir:    Directory where extracted JPEGs are saved.
        tmp_dir:       /tmp directory for split download cache.

    Returns:
        dict mapping (sid, role) → Path of saved JPEG.
        Partial result if some members not found (caller should log misses).

    Why one streaming pass per stem: tarfile r| mode cannot seek or rewind.
    All needed members must be extracted in a single sequential scan.
    """
    split_files = _IMAGE_TAR_SPLITS.get(stem, [])
    if not split_files:
        log.warning(f"extract_images_streaming: no splits for stem {stem!r}")
        return {}

    # Build a fast lookup: member_path → (sid, role)
    member_lookup: dict = {}   # str member path → (sid, role)
    for variants, label in needed.items():
        for v in variants:
            member_lookup[v] = label

    found = {}
    remaining = len(needed)

    reader = SplitStreamReader(split_files, hf_dataset_id, tmp_dir)
    try:
        with tarfile.open(fileobj=reader, mode="r|") as tf:
            for member in tf:
                if not member.isfile():
                    continue
                label = member_lookup.get(member.name)
                if label is None:
                    continue   # not needed — tarfile skips content automatically

                try:
                    fobj = tf.extractfile(member)
                    if fobj is None:
                        continue
                    raw = fobj.read()
                    img = Image.open(io.BytesIO(raw)).convert("RGB")
                    sid, role = label
                    suffix    = "source" if role == "src" else "target"
                    save_path = images_dir / f"{sid}_{suffix}.jpg"
                    img.save(save_path, "JPEG", quality=95)
                    found[label] = save_path
                    remaining -= 1
                    if remaining == 0:
                        log.info(f"All {len(needed)} images found — stopping early")
                        break
                except Exception as e_img:
                    log.debug(f"Image decode failed for {member.name}: {e_img}")
    finally:
        reader.close()   # deletes current split from /tmp

    log.info(
        f"extract_images_streaming({stem!r}): "
        f"{len(found)}/{len(needed)} found"
    )
    return found


# ── Image fetch from HuggingFace (path-based) ─────────────────────────────────


from huggingface_hub import HfFileSystem
_hf_fs = None  # lazy-initialised

def _get_hf_fs() -> HfFileSystem:
    global _hf_fs
    if _hf_fs is None:
        _hf_fs = HfFileSystem()
    return _hf_fs

def fetch_image_from_hf(
    hf_repo_id: str,
    image_path: str,
    tar_cache_dir: Optional[Path] = None,
) -> Optional[Image.Image]:
    """Fetch a single image from a HuggingFace dataset repo by its path.

    Confirmed §6.1: Strategies 1 and 2 BOTH fail for all sysuyy/ImgEdit paths.
    Images are stored in tar archives — Strategy 3 is the primary path.

    Strategies (in order):
      1. HfFileSystem direct open — fast if image is not LFS-packed.
      2. hf_hub_download exact path — works for a small minority of files.
      3. Tar archive extraction — confirmed-working primary method.
         Uses _IMAGE_TAR_NAMES / _IMAGE_TAR_LOCAL built by build_image_tar_names().
         Downloads only the one specific tar archive needed (lazy, cached).
         Tries multiple member path variants (full / prefix-stripped / basename).

    Args:
        hf_repo_id:     HuggingFace dataset repo ID.
        image_path:     Path from parquet, e.g.
                        "results_background_laion_part5/00017_00003_000035901/original.png"
        tar_cache_dir:  Optional dir for tar file cache (defaults to HF cache).
    """
    if not isinstance(image_path, str):
        image_path = str(image_path)

    # Strategy 1: direct HfFileSystem open
    fs = _get_hf_fs()
    hf_path = f"datasets/{hf_repo_id}/{image_path}"
    try:
        with fs.open(hf_path, "rb") as fh:
            data = fh.read()
        return Image.open(io.BytesIO(data))
    except Exception as e1:
        log.debug(f"HfFileSystem open failed for {hf_path}: {e1}")

    # Strategy 2: hf_hub_download exact path
    try:
        local = hf_hub_download(
            repo_id=hf_repo_id, repo_type="dataset", filename=image_path
        )
        return Image.open(local)
    except Exception as e2:
        log.debug(f"hf_hub_download failed for {image_path}: {e2}")

    # Strategy 3: single-image streaming (for ad-hoc fetches only)
    # For bulk download use extract_images_streaming() in download_imgedit_subset().
    # Single-image fetch streams through the tar until the member is found.
    # WARNING: for large tars (9 splits × 42.9 GB) this may scan up to 357 GB
    # before finding your image if it's near the end. Only practical for small
    # tars (1-2 splits) or when the image is expected near the start of the tar.
    path_parts = image_path.replace("\\", "/").split("/")
    if len(path_parts) >= 2:
        prefix = path_parts[0]
        member_candidates = [
            image_path,               # full path: results_X/sample_id/original.png
            "/".join(path_parts[1:]), # no-prefix: sample_id/original.png
            path_parts[-1],           # basename: original.png
        ]
        stems = _find_tar_stems_for_prefix(prefix)
        for stem in stems:
            split_files = _IMAGE_TAR_SPLITS.get(stem, [])
            if not split_files:
                continue
            try:
                reader = SplitStreamReader(
                    split_files, hf_repo_id,
                    tmp_dir=Path("/tmp/fetch_single_tmp")
                )
                found_img = None
                with tarfile.open(fileobj=reader, mode="r|") as tf:
                    for member in tf:
                        if not member.isfile():
                            continue
                        if member.name in member_candidates:
                            fobj = tf.extractfile(member)
                            if fobj:
                                found_img = Image.open(io.BytesIO(fobj.read()))
                            break
                reader.close()
                if found_img is not None:
                    return found_img
            except Exception as e3:
                log.debug(f"Streaming Strategy 3 failed for {image_path}: {e3}")

    log.warning(f"Could not fetch image via any strategy: {image_path}")
    return None


# ── Mask fetch from ImgEdit_recap_mask ────────────────────────────────────
# _load_mask_for_sample() is ready to use — §2.3 confirmed mask schema.
# MASK_INNER_RLE_FIELD='mask', MASK_BOX_FORMAT='xyxy' set in §2.4.

# ── Mask field constants — sourced from §2.4 confirmed schema ────────────
# NOTE: field names are defined in §2.4. These mirror them here for local
# reference in §3.1 functions. Do NOT hardcode field names anywhere else.
#
# MASK_JOIN_FIELD      = "path"          (confirmed §2.3)
# MASK_SEG_FIELD       = "segmentation"  (confirmed §2.3)
# MASK_INNER_RLE_FIELD = <set in §2.4 after step 9>
# MASK_BOX_FORMAT      = <set in §2.4 after step 9>
#
# build_mask_index() / _index_mask_record() will raise RuntimeError if
# §2.4 constants are not set — fail-fast before any training loop runs.

def _load_mask_for_sample(
    mask_index: dict,
    sample_id: str,
) -> dict:
    """Look up mask annotations for a given sample_id.

    Args:
        mask_index: dict built by build_mask_index(), mapping sample_id → annotation data.
        sample_id:  Produced by extract_sample_id() from an ImgEdit image path.
                    For mask-dataset samples this equals normalize_mask_path_to_sample_id()
                    applied to the mask JSON's 'path' field.

    Returns:
        {"annotations": list_of_annotation_dicts, "resolution": {"height": int, "width": int}}
        Each annotation dict has:
            "class_name":   str
            "bbox":         list[4 float] — coords in box_format (confirm from §2.4)
            "segmentation": COCO RLE dict ({"counts": ..., "size": [H, W]})
                            — present only if MASK_INNER_RLE_FIELD was set in §2.4
        Returns {"annotations": [], "resolution": {}} for unmatched samples
        (e.g. action-partition video frames, which are not in the mask dataset).

    Why no guard on MASK_INNER_RLE_FIELD here: the index was already built
    with _index_mask_record(), which raised at build time if §2.4 was not set.
    """
    entry = mask_index.get(sample_id)
    if entry is None:
        return {"annotations": [], "resolution": {}}
    return entry


def build_mask_index(
    mask_tar_files: list,
    hf_mask_id: str = None,
) -> dict:
    """Load all mask JSON tars into a dict keyed by sample_id.

    Dataset format: jsons/partN.tar archives (NOT parquet).
    Each tar contains JSON or JSONL annotation files, one entry per image.

    Args:
        mask_tar_files: local paths to downloaded mask tar files (jsons/partN.tar).
                        Pass [] and set hf_mask_id to auto-download from HuggingFace.
        hf_mask_id: if provided, download jsons/*.tar from this HF dataset first.

    Returns dict: {sample_id: {"annotations": [{"bbox": [...], "segmentation": {...}}]}}

    REQUIRES: §2.4 MASK_INNER_RLE_FIELD must be set (confirmed = 'mask').
    Confirmed from §2.3 step 9 — safe to call now that §2.4 is populated.

    Why tar iteration: the mask dataset has no parquet files; all annotations
    live in tar archives containing individual JSON files per sample.
    """
    import tarfile, io

    # Guard: ensure §2.4 constants are set before building the index.
    # MASK_INNER_RLE_FIELD being None means step 9 hasn't been reviewed yet.
    # Annotations without RLE can still be indexed (bbox-only), but raise
    # here so the caller knows masks are incomplete.
    if MASK_INNER_RLE_FIELD is None:
        raise RuntimeError(
            "MASK_INNER_RLE_FIELD is not set (§2.4). "
            "Run §2.3 step 9, read the output, then set MASK_INNER_RLE_FIELD in §2.4 "
            "before calling build_mask_index()."
        )

    # ── Auto-download jsons/*.tar from HuggingFace if not provided locally ─
    if hf_mask_id and not mask_tar_files:
        from huggingface_hub import hf_hub_download, HfApi
        api = HfApi()
        all_files = list(api.list_repo_files(hf_mask_id, repo_type="dataset"))
        json_tar_names = sorted(f for f in all_files if f.startswith("jsons/") and f.endswith(".tar"))
        if not json_tar_names:
            raise RuntimeError(
                f"No jsons/*.tar files found in {hf_mask_id}. "
                "Dataset structure may have changed — re-run §2.3 to inspect."
            )
        log.info(
            f"Downloading {len(json_tar_names)} JSON tars from {hf_mask_id}. "
            f"This may take several minutes — tars are ~50-450 MB each."
        )
        mask_tar_files = [
            hf_hub_download(repo_id=hf_mask_id, repo_type="dataset", filename=f)
            for f in json_tar_names
        ]

    if not mask_tar_files:
        raise ValueError(
            "mask_tar_files is empty and hf_mask_id was not provided. "
            "Pass hf_mask_id=CFG.mask_dataset_id to auto-download, or provide local paths."
        )

    index = {}
    for tar_path in mask_tar_files:
        log.info(f"Indexing mask tar: {tar_path}")
        with tarfile.open(tar_path, "r:*") as tf:
            members = tf.getmembers()
            json_members  = [m for m in members if m.name.endswith(".json")]
            jsonl_members = [m for m in members if m.name.endswith(".jsonl")]

            if json_members:
                # Individual .json file per sample
                for m in json_members:
                    fobj = tf.extractfile(m)
                    if fobj is None:
                        continue
                    try:
                        entry = json.loads(fobj.read().decode("utf-8"))
                    except Exception as exc:
                        log.warning(f"JSON parse error in {m.name}: {exc}")
                        continue
                    entries = entry if isinstance(entry, list) else [entry]
                    for record in entries:
                        _index_mask_record(record, index)

            elif jsonl_members:
                # Single JSONL file — one JSON object per line
                for m in jsonl_members:
                    fobj = tf.extractfile(m)
                    if fobj is None:
                        continue
                    for line in fobj:
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            record = json.loads(line)
                        except Exception as exc:
                            log.warning(f"JSONL parse error in {m.name}: {exc}")
                            continue
                        _index_mask_record(record, index)

            else:
                log.warning(
                    f"No .json or .jsonl members found in {tar_path} — "
                    "skipping. Re-run §2.3 if mask dataset structure changed."
                )

    log.info(f"Mask index built: {len(index):,} entries from {len(mask_tar_files)} tars")
    return index


def _index_mask_record(record: dict, index: dict) -> None:
    """Parse one mask annotation JSON document and insert into index.

    Confirmed mask JSON structure (§2.3):
        {
          "path": "00027/00038/000386475.jpg",  ← join key after normalization
          "resolution": {"height": H, "width": W},
          "segmentation": {
              "object":     [{"class_name": str, "bbox": [x,y,x,y], <rle_field>: {...}}, ...],
              "background": [{"class_name": str, "bbox": [x,y,x,y], <rle_field>: {...}}, ...],
              "box_format": "xyxy"   ← or "xywh" — confirm from §2.4 MASK_BOX_FORMAT
          },
          "obj_count": {"class": count, ...},
          ...
        }

    Both 'object' and 'background' annotation lists are indexed (Phase 2
    inpainting may need background masks for global edits).

    Separated from build_mask_index so JSON-per-file and JSONL paths use
    identical parsing logic — guards against train/inference divergence.

    BLOCKED: call build_mask_index() only after §2.4 MASK_INNER_RLE_FIELD is set.
    """
    if not isinstance(record, dict):
        return

    # ── Join key: normalize 'path' field ──────────────────────────────────
    path_raw = record.get(MASK_JOIN_FIELD)   # "path" — confirmed §2.3
    if path_raw is None:
        log.debug("Mask record missing 'path' field — skipping.")
        return
    sid = normalize_mask_path_to_sample_id(str(path_raw))

    # ── Resolution ────────────────────────────────────────────────────────
    resolution = record.get(MASK_RESOLUTION_FIELD, {}) or {}

    # ── Parse all annotation objects from both regions ─────────────────────
    seg_dict  = record.get(MASK_SEG_FIELD, {}) or {}
    box_fmt   = seg_dict.get(MASK_BOX_FORMAT_KEY)  # e.g. "xyxy"
    annotations: list = []

    for region_key in [MASK_OBJ_LIST_KEY, MASK_BG_LIST_KEY]:
        for raw_ann in (seg_dict.get(region_key) or []):
            if not isinstance(raw_ann, dict):
                continue
            ann: dict = {
                "class_name": raw_ann.get(MASK_CLASS_NAME_KEY, ""),
                "region":     region_key,  # "object" or "background"
            }

            # ── Bbox ──────────────────────────────────────────────────────
            bbox_raw = raw_ann.get(MASK_BBOX_KEY)
            if bbox_raw is not None:
                ann["bbox"] = list(bbox_raw) if hasattr(bbox_raw, "__iter__") else [bbox_raw]

            # ── RLE mask ──────────────────────────────────────────────────
            # MASK_INNER_RLE_FIELD is set in §2.4 after step 9 inspection.
            # If still None, annotation is stored without a mask field —
            # decode_rle_mask() will skip it safely during training.
            if MASK_INNER_RLE_FIELD is not None:
                rle_raw = raw_ann.get(MASK_INNER_RLE_FIELD)
                if rle_raw is not None:
                    if isinstance(rle_raw, str):
                        # Confirmed §2.3: 'mask' field is a compressed COCO RLE
                        # string (NOT JSON-encoded). Build a proper COCO RLE dict
                        # pairing it with the image dimensions from 'resolution'.
                        # decode_rle_mask() handles str→bytes conversion internally.
                        H = resolution.get("height", 0)
                        W = resolution.get("width", 0)
                        if H == 0 or W == 0:
                            log.debug(
                                f"Missing resolution for {sid!r} — "
                                f"RLE size will be [0,0] (mask unusable)"
                            )
                        ann["segmentation"] = {
                            "counts": rle_raw,
                            "size": [H, W],   # required by pycocotools.mask.decode
                        }
                    elif isinstance(rle_raw, dict):
                        # Future-proofing: handle pre-formed COCO RLE dicts.
                        ann["segmentation"] = rle_raw
                    else:
                        log.debug(
                            f"Unexpected RLE type {type(rle_raw)} "
                            f"for sample_id={sid!r} class={ann['class_name']!r}"
                        )

            # ── Optional quality scores (confirmed present §2.3 step 9) ──────
            score_raw = raw_ann.get(MASK_SCORE_KEY)
            if score_raw is not None:
                ann["score"] = float(score_raw[0]) if isinstance(score_raw, list) else float(score_raw)
            clip_raw = raw_ann.get(MASK_CLIP_SCORE_KEY)
            if clip_raw is not None:
                ann["clip_score"] = float(clip_raw)
            aes_raw = raw_ann.get(MASK_AES_SCORE_KEY)
            if aes_raw is not None:
                ann["aes_score"] = float(aes_raw)

            annotations.append(ann)

    index[sid] = {
        "annotations": annotations,
        "resolution":  resolution,
        "box_format":  box_fmt,
    }


# ── File I/O helpers ─────────────────────────────────────────────────────
def save_json(obj: Any, path: Path, indent: int = 2) -> None:
    """Atomic JSON write (write tmp → rename)."""
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=indent, ensure_ascii=False))
    tmp.rename(path)

def load_json(path: Path) -> Any:
    if not path.exists():
        raise FileNotFoundError(f"Expected JSON not found: {path}")
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError as e:
        raise ValueError(f"Corrupt JSON at {path}: {e}") from e


print("Utilities loaded:")
print("  infer_edit_type_from_filename, extract_sample_id")
print("  decode_rle_mask, route_bbox")
print("  fetch_image_from_hf (HfFileSystem + hf_hub_download fallback)")
print("  build_mask_index, _load_mask_for_sample")
print("  save_json, load_json")
print()
print("§3.1 utilities loaded (v2.9).")
print("  SplitStreamReader: downloads 1 split at a time to /tmp, never to Drive")
print("  extract_images_streaming(): one streaming pass per tar stem")
print("  Peak disk: /tmp ~42.9 GB (1 split) + Drive: JPEGs only")
print("  build_image_tar_names(), _find_tar_stems_for_prefix() (4-rule)")
print("  §2.3 confirmed: MASK_INNER_RLE_FIELD=\'mask\', MASK_BOX_FORMAT=\'xyxy\'")


## §4 — Phase 0: Dataset Download
`download_imgedit_subset()` streams ImgEdit, samples `n` examples with a fixed seed,  
saves source/target images as JPEG + raw segmentation JSON, and writes `samples.json`.  
**Re-runnable:** if `samples.json` already exists and has the right count, skips download.


In [ ]:
# ── §4.1  download_imgedit_subset ─────────────────────────────────────────
# Architecture (v2.5 — tar-based image fetch, confirmed working schemas):
#
#   sysuyy/ImgEdit parquet files  ─► metadata (paths + prompts, no images)
#       ↓ edit_type inferred from parquet filename prefix
#       ↓ sample_id extracted from input_images path component [1]
#       ↓ nested-struct parquets handled automatically (§6.1 fix)
#   HuggingFace image tar archives ─► JPEG images (tar extraction, §6.1 confirmed)
#       Strategy 3 in fetch_image_from_hf — builds tar index from §2.5 listing
#       Only the specific tar archives needed are downloaded (lazy, Drive-cached)
#   sysuyy/ImgEdit_recap_mask     ─► RLE masks + bboxes (join on sample_id)
#       ↓
#   samples.json manifest         ─► consumed by audit_dataset + build_vlm_dataset
#
# §2.3 COMPLETE: MASK_INNER_RLE_FIELD='mask', MASK_BOX_FORMAT='xyxy' confirmed.
# §6.1 FIXES:    nested-struct parquets, numpy path extraction, tar image fetch.
#   load_masks=True ready — set STAGE="B" in §7.1 after smoke test passes.

import os, io, json
import tarfile
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional
from tqdm.auto import tqdm
from PIL import Image
from huggingface_hub import HfApi, hf_hub_download

def _extract_path_str(val) -> str:
    """Robustly extract a single path string from a parquet field.

    Handles: str, list, tuple, numpy.ndarray (all confirmed in sysuyy/ImgEdit).
    Confirmed §6.1: input_images is list[str] in pandas (numpy object array
    or Python list), so plain str(val) would produce "['path']" — wrong.
    """
    if isinstance(val, str):
        return val
    if isinstance(val, (list, tuple)) and len(val) > 0:
        return _extract_path_str(val[0])
    if isinstance(val, np.ndarray) and val.size > 0:
        return _extract_path_str(val.flat[0])
    return str(val)  # fallback: best-effort

def download_imgedit_subset(
    n: int                    = CFG.n_subset,
    output_dir: Path          = CFG.data_dir,
    load_masks: bool          = True,   # set True after §2.3 confirms schema
    edit_type_filter: Optional[list] = None,   # None = all types
    force: bool               = False,
) -> Path:
    """Download n samples from sysuyy/ImgEdit + masks from ImgEdit_recap_mask.

    Args:
        n:                Number of samples to collect (first n rows in parquet order).
        output_dir:       Drive path for all outputs (persists across sessions).
        load_masks:       If True, join with ImgEdit_recap_mask. Requires §2.3 done.
        edit_type_filter: If set, only include these edit types (e.g. ["add","remove"]).
        force:            Re-download even if manifest already complete.

    Returns:
        Path to samples.json manifest.

    Raises:
        RuntimeError: if <n//2 samples saved (streaming/auth failure).
        RuntimeError: if load_masks=True but mask schema not yet confirmed.
    """
    if load_masks and MASK_INNER_RLE_FIELD is None:
        raise RuntimeError(
            "load_masks=True requires MASK_INNER_RLE_FIELD to be set (§2.4).\n"
            "MASK_INNER_RLE_FIELD should be \'mask\' — confirmed from §2.3 step 9.\n"
            "Run §2.4 cell to set the constant before calling with load_masks=True."
        )

    manifest_path = output_dir / "samples.json"

    # ── Restart-safety ────────────────────────────────────────────────────
    if manifest_path.exists() and not force:
        existing = load_json(manifest_path)
        if len(existing) >= n * 0.95:
            log.info(f"Manifest exists with {len(existing):,} samples. Skipping. (force=True to redo)")
            return manifest_path
        log.warning(f"Incomplete manifest ({len(existing):,} < {n:,}). Re-downloading.")

    output_dir.mkdir(parents=True, exist_ok=True)
    images_dir = output_dir / "images"
    images_dir.mkdir(exist_ok=True)
    segs_dir   = output_dir / "segmentations"
    segs_dir.mkdir(exist_ok=True)

    api = HfApi()

    # ── Step 0: build split-tar index (lazy, no-op if already done) ─────────
    # v2.9: streaming extraction — no tar ever assembled to disk.
    # SplitStreamReader downloads 1 split (~42.9 GB) to /tmp at a time.
    # Peak Drive disk: extracted JPEGs only. Peak /tmp: ~42.9 GB.
    log.info("Building split-tar index from HF repo ...")
    build_image_tar_names(hf_dataset_id=CFG.hf_dataset_id)
    if not _IMAGE_TAR_SPLITS:
        log.warning("No split tars found. Re-run §2.5 to inspect repo structure.")

    tmp_dir = Path("/tmp/split_stream_tmp")   # Colab local disk for split parts
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # ── Known unavailable path prefixes (confirmed §2.5 — no matching tar) ──
    KNOWN_UNAVAILABLE_PREFIXES = {
        "results_compose_part0",     # no tar in repo
        "results_extract_ref_part1", # no tar in repo (reference edit type)
    }

    # ── Step 1: list parquet files from main dataset ───────────────────────
    log.info(f"Listing parquet files from {CFG.hf_dataset_id} ...")
    all_files = api.list_repo_files(CFG.hf_dataset_id, repo_type="dataset")
    parquet_files = sorted([f for f in all_files if f.endswith(".parquet")])
    log.info(f"Found {len(parquet_files)} parquet files")

    # Apply edit_type filter
    if edit_type_filter:
        parquet_files = [
            f for f in parquet_files
            if infer_edit_type_from_filename(f) in set(edit_type_filter)
        ]
        log.info(f"After edit_type_filter={edit_type_filter}: {len(parquet_files)} files")

    if not parquet_files:
        raise RuntimeError("No parquet files found — check CFG.hf_dataset_id and edit_type_filter.")

    # ── Step 2: load all parquet metadata (lightweight — no images) ────────
    log.info("Loading parquet metadata (text only, no image download) ...")
    all_rows = []
    for pq_file in tqdm(parquet_files, desc="Loading parquets", unit="file"):
        try:
            local_pq = hf_hub_download(
                repo_id=CFG.hf_dataset_id, repo_type="dataset", filename=pq_file
            )
            # ── Try standard columnar schema first ─────────────────────────────
            # Some parquets (e.g. content_memory, version_backtracking) use a
            # nested list-of-structs schema where columns are NOT top-level fields.
            # Confirmed §6.1: FieldRef.Name(input_images) fails for these files.
            try:
                df = pq.read_table(local_pq, columns=[
                    COL_INPUT_IMAGES, COL_OUTPUT_IMAGES, COL_PROMPT
                ]).to_pandas()
            except Exception as e_col:
                if "FieldRef.Name" not in str(e_col) and "No match" not in str(e_col):
                    raise   # unexpected error — re-raise
                # ── Nested list-of-structs format ──────────────────────────────
                # Schema: single column of type list<struct<input_images, output_images, prompt>>
                # Fix: read all columns, find the struct column, explode into rows.
                log.debug(f"Nested struct schema in {pq_file} — applying unnest")
                table_raw = pq.read_table(local_pq)
                df_raw    = table_raw.to_pandas()
                data_cols = [c for c in df_raw.columns if not c.startswith("__")]
                if not data_cols:
                    raise ValueError(f"No data columns found in {pq_file}")
                col0 = data_cols[0]
                # Explode: each row in col0 is a list of record dicts
                df_exploded = df_raw.explode(col0).reset_index(drop=True)
                nested_vals = df_exploded[col0].dropna()
                if len(nested_vals) == 0:
                    raise ValueError(f"Empty after explode in {pq_file}")
                if isinstance(nested_vals.iloc[0], dict):
                    df = pd.DataFrame(nested_vals.tolist())
                else:
                    raise ValueError(
                        f"Unexpected nested type {type(nested_vals.iloc[0])} in {pq_file}"
                    )
                # Validate expected columns exist after unnesting
                missing = [c for c in [COL_INPUT_IMAGES, COL_OUTPUT_IMAGES, COL_PROMPT]
                           if c not in df.columns]
                if missing:
                    raise ValueError(
                        f"Missing columns {missing} after unnesting {pq_file}. "
                        f"Available: {list(df.columns)}"
                    )
                log.info(f"Unnested nested-struct parquet {pq_file}: {len(df)} rows")

            df["_pq_file"]    = pq_file
            df["_edit_type"]  = infer_edit_type_from_filename(pq_file)
            all_rows.append(df)
        except Exception as e:
            log.warning(f"Failed to load {pq_file}: {e}")

    full_df = pd.concat(all_rows, ignore_index=True)
    log.info(f"Total metadata rows: {len(full_df):,}")

    # ── Step 3: take the first n rows in parquet file order ────────────────
    # Rationale (v2.9): random sampling scattered needed images across ALL tar
    # stems, forcing a full scan of every split (~43 GB each). Taking the
    # first n rows instead concentrates images in the earliest tar splits so
    # streaming extraction terminates early — much faster under time constraints.
    # Parquet files are loaded in sorted order (see parquet_files above), so
    # this is deterministic and restart-safe without needing a seed.
    type_counts = full_df["_edit_type"].value_counts()
    log.info(f"Edit type distribution in full metadata:\n{type_counts.to_string()}")

    sample_df = full_df.head(n).reset_index(drop=True)
    sampled_types = sample_df["_edit_type"].value_counts()
    log.info(
        f"Taking first {len(sample_df):,} rows in parquet order (target={n:,}).\n"
        f"Edit type distribution in sample:\n{sampled_types.to_string()}"
    )

    # ── Step 4: build mask index if requested ─────────────────────────────
    mask_index = {}
    if load_masks:
        log.info(f"Building mask index from {CFG.mask_dataset_id} ...")
        mask_index = build_mask_index(
            mask_tar_files=[],           # empty — build_mask_index downloads from HF
            hf_mask_id=CFG.mask_dataset_id,
        )
        log.info(f"Mask index: {len(mask_index):,} entries")

    # ── Step 5: group samples by tar stem, then process-and-delete ────────
    # ⚠ DISK CONSTRAINT (confirmed §2.6): one tar stem ≈ 43 GB assembled.
    # Strategy: process all samples from ONE tar at a time, delete tar after,
    # then move to the next. Peak disk = 1 reassembled tar + extracted JPEGs.
    # With 235 GB available, persistent caching of all tars is NOT viable.
    #
    # Algorithm:
    #  1. For each sampled row, find matching tar stems for its image path prefix.
    #  2. Group rows by their PRIMARY tar stem (first matching stem).
    #  3. Rows with NO stem (unavailable prefix): log + skip immediately.
    #  4. Per stem: assemble tar → extract all needed images → DELETE tar.

    manifest       = []
    error_log      = []
    n_mask_hits, n_mask_misses = 0, 0
    sample_counter = 0   # monotonic; stays consistent across tar groups

    # ── 5a: Classify each row by primary tar stem ──────────────────────────
    stem_to_rows: dict = {}
    for _, row in sample_df.iterrows():
        inp_path   = _extract_path_str(row[COL_INPUT_IMAGES])
        out_path   = _extract_path_str(row[COL_OUTPUT_IMAGES])
        inp_prefix = inp_path.split("/")[0] if "/" in inp_path else inp_path

        if inp_prefix in KNOWN_UNAVAILABLE_PREFIXES:
            error_log.append({"reason": "no_tar_for_prefix", "prefix": inp_prefix})
            continue

        stems = _find_tar_stems_for_prefix(inp_prefix)
        if not stems:
            log.warning(f"No tar stems for prefix {inp_prefix!r} — skipping")
            error_log.append({"reason": "no_tar_stems", "prefix": inp_prefix})
            continue

        primary_stem = stems[0]
        stem_to_rows.setdefault(primary_stem, []).append({
            "row": row, "inp_path": inp_path, "out_path": out_path,
            "inp_prefix": inp_prefix, "all_stems": stems,
        })

    total_stems = len(stem_to_rows)
    log.info(
        f"Tar stem groups: {total_stems} stems, "
        f"{sum(len(v) for v in stem_to_rows.values())} rows. "
        f"Peak disk ≈ 43 GB (1 tar at a time)."
    )

    # ── 5b: Stream through each tar stem — extract images — no tar on disk ────
    # v2.9 streaming design:
    #   SplitStreamReader downloads split.000 → reads → deletes → split.001 → ...
    #   tarfile(mode="r|") scans sequentially; extracts only needed members.
    #   All needed images from a stem are found in ONE streaming pass.
    #   Peak /tmp: 1 split file (~42.9 GB). Peak Drive: JPEGs only.

    for stem_idx, (primary_stem, row_dicts) in enumerate(stem_to_rows.items(), 1):
        log.info(
            f"[{stem_idx}/{total_stems}] Streaming stem '{primary_stem}' — "
            f"{len(row_dicts)} sample(s)"
        )
        split_files = _IMAGE_TAR_SPLITS.get(primary_stem, [])
        if not split_files:
            log.warning(f"No split files for stem '{primary_stem}' — skipping")
            for rd in row_dicts:
                error_log.append({"reason": "no_split_files", "stem": primary_stem})
            continue

        log.info(
            f"  {len(split_files)} split(s) × ~42.9 GB each. "
            f"Scanning sequentially (no full tar assembled)."
        )

        # Build needed-member lookup for this stem:
        # needed_map: frozenset(path_variants) → (sid, "src"|"tgt")
        needed_map: dict = {}
        row_meta: dict   = {}   # sid → row info for building manifest later

        for rd in row_dicts:
            sample_counter += 1
            sid      = f"{sample_counter:07d}"
            inp_path = rd["inp_path"]
            out_path = rd["out_path"]
            row      = rd["row"]

            def _variants(p):
                pp = p.replace("\\", "/").split("/")
                return frozenset([p, "/".join(pp[1:]) if len(pp)>1 else p, pp[-1]])

            needed_map[_variants(inp_path)] = (sid, "src")
            needed_map[_variants(out_path)] = (sid, "tgt")
            row_meta[sid] = {
                "row": row, "inp_path": inp_path, "out_path": out_path,
                "hf_sample_id": extract_sample_id(inp_path),
            }

        # One streaming pass through all splits for this stem
        found_images = extract_images_streaming(
            stem          = primary_stem,
            needed        = needed_map,
            hf_dataset_id = CFG.hf_dataset_id,
            images_dir    = images_dir,
            tmp_dir       = tmp_dir,
        )
        # found_images: (sid, role) → Path of saved JPEG

        # Assemble manifest entries
        sids_with_both = set()
        for (sid, role), save_path in found_images.items():
            sids_with_both.add(sid)   # track which sids have at least one image

        for sid, meta in row_meta.items():
            src_key = (sid, "src")
            tgt_key = (sid, "tgt")
            src_path = found_images.get(src_key)
            tgt_path = found_images.get(tgt_key)

            if src_path is None or tgt_path is None:
                error_log.append({
                    "sid": sid, "reason": "image_not_in_tar",
                    "inp": meta["inp_path"], "stem": primary_stem,
                    "src_found": src_path is not None,
                    "tgt_found": tgt_path is not None,
                })
                sample_counter -= 1   # don't count failed samples
                continue

            row      = meta["row"]
            seg_path = segs_dir / f"{sid}_seg.json"
            hf_sid   = meta["hf_sample_id"]
            mask_data = (mask_index.get(hf_sid, {"annotations": []})
                         if load_masks else {"annotations": []})
            if load_masks:
                if mask_data["annotations"]: n_mask_hits += 1
                else:                        n_mask_misses += 1
            save_json(mask_data, seg_path)

            # Get image dimensions from saved JPEG
            try:
                with Image.open(src_path) as img_check:
                    W, H = img_check.size
            except Exception:
                W, H = 0, 0

            manifest.append({
                "sample_id":        sid,
                "hf_sample_id":     hf_sid,
                "pq_file":          row["_pq_file"],
                "edit_type":        row["_edit_type"],
                "source_image":     str(src_path.relative_to(output_dir)),
                "target_image":     str(tgt_path.relative_to(output_dir)),
                "segmentation":     str(seg_path.relative_to(output_dir)),
                "edit_instruction": str(row[COL_PROMPT] or ""),
                "img_width":        W,
                "img_height":       H,
                "mask_loaded":      load_masks and bool(mask_data["annotations"]),
                "shard_id":         None,
                "row_index":        None,
                "mask_path":        None,
            })

            if len(manifest) % 500 == 0:
                save_json(manifest, manifest_path)
                log.info(f"Checkpoint: {len(manifest):,} samples")

        log.info(
            f"[{stem_idx}/{total_stems}] '{primary_stem}' done — "
            f"{len(manifest):,} total samples"
        )

    # ── Save manifest + error log ──────────────────────────────────────────
    save_json(manifest, manifest_path)
    log.info(f"Manifest: {len(manifest):,} samples → {manifest_path}")

    if error_log:
        err_path = output_dir / "download_errors.json"
        save_json(error_log, err_path)
        log.warning(f"{len(error_log)} errors → {err_path}")

    if load_masks:
        log.info(f"Mask hits: {n_mask_hits:,}  Misses: {n_mask_misses:,}")
        miss_rate = n_mask_misses / max(n_mask_hits + n_mask_misses, 1)
        if miss_rate > 0.10:
            log.warning(
                f"Mask miss rate {miss_rate:.1%} > 10%. "
                f"Verify hf_sample_id join key with §2.3 output."
            )

    # ── Failure threshold ────────────────────────────────────────────────────
    # For small n (smoke test), require >0 samples; for large n, require >50%.
    # §6.1: "Only 0 samples saved (target 5)" — ALL images failed because
    #       direct HF fetch doesn't work. After tar fix this should pass.
    min_required = 1 if n < 10 else n // 2
    fail_rate = 1.0 - len(manifest) / max(n, 1)
    if len(manifest) < min_required:
        raise RuntimeError(
            f"Only {len(manifest):,} samples saved (target {n:,}, "
            f"minimum {min_required:,}). fail_rate={fail_rate:.0%}.\n"
            f"Check download_errors.json for details.\n"
            f"If images fail to fetch: (a) run §2.5 to inspect repo structure, "
            f"(b) verify _IMAGE_TAR_NAMES is populated (build_image_tar_names()), "
            f"(c) check fetch_image_from_hf Strategy 3 in §3.1."
        )
    if fail_rate > 0.10:
        log.warning(f"High image fetch failure rate: {fail_rate:.0%} of {n} samples")

    print(f"\n✓ download_imgedit_subset: {len(manifest):,} samples saved.")
    if not load_masks:
        print("  NOTE: masks NOT loaded (load_masks=False). Run again with load_masks=True")
        print("        after completing §2.3 and setting MASK_SAMPLE_ID_FIELD in §3.1.")
    return manifest_path


print("download_imgedit_subset() defined (v2.9 — streaming extraction).")
print("  Peak /tmp: 1 split file (~42.9 GB) at a time; peak Drive: JPEGs only")
print("  SplitStreamReader: downloads split → reads → deletes → next split")
print("  One streaming tar pass per stem; extracts all needed members in sequence")
print("  Skips: results_compose_part0, results_extract_ref_part1 (no tar)")
print("  Checkpoint every 500 samples; restart-safe")
print(f"  n_subset = {CFG.n_subset:,} (increase in §0.1 for larger run)")


## §5 — Phase 0: Dataset Audit (`audit_dataset`)
Run after `download_imgedit_subset()`. Gates training on data quality.  
**Decision gate:** if `invalid_rle_masks > 5%` or `fallback_to_full_image` for any of  
{add, remove, replace} exceeds `15%` → inspect before proceeding to Phase 1.


In [ ]:
# ── §5.1  audit_dataset ───────────────────────────────────────────────────
# v2 updates:
#   - Reads "mask_loaded" flag from manifest (new field)
#   - Reports mask hit rate (separate from invalid_rle — masks may not be
#     loaded yet if §2.3 is incomplete)
#   - Handles new edit types: action, content, hybrid, reference, version
#   - Same gate thresholds as v1; gate only fires when masks are loaded

import json, logging
from collections import defaultdict
from pathlib import Path
from typing import Optional
from tqdm.auto import tqdm

log = logging.getLogger("pipeline")


def audit_dataset(
    raw_data_dir: Path          = CFG.data_dir,
    sample_limit: Optional[int] = None,
) -> dict:
    """Formal pre-training data quality audit.

    Run AFTER download_imgedit_subset() and BEFORE build_vlm_dataset().
    Surfaces counts that would otherwise become silent training noise.

    The RLE gate is only enforced when mask_loaded=True on samples.
    If masks were not loaded (load_masks=False in download), invalid_rle_masks
    will equal total_samples and the gate will NOT fire (it is deferred).
    Always load masks before final Phase 1 training.

    Returns dict with spec §3.2 required fields plus v2 additions:
        total_samples
        missing_source_images, missing_target_images  (+ _ids lists)
        masks_not_loaded         int   — samples where mask_loaded=False
        invalid_rle_masks        int   — among loaded masks
        fallback_to_full_image_by_type
        edit_type_distribution
        _gate_passed             bool
        _gate_messages           list[str]
    """
    manifest_path = raw_data_dir / "samples.json"
    manifest      = load_json(manifest_path)

    if sample_limit:
        manifest = manifest[:sample_limit]
        log.info(f"audit_dataset: limited to {sample_limit} samples")

    total = len(manifest)
    log.info(f"Auditing {total:,} samples ...")

    missing_src_ids   = []
    missing_tgt_ids   = []
    invalid_rle_ids   = []
    masks_not_loaded  = 0

    edit_type_counts   = defaultdict(int)
    fallback_by_type   = defaultdict(int)
    fallback_ids_by_type = defaultdict(list)

    for entry in tqdm(manifest, desc="Auditing", unit="sample"):
        sid = entry["sample_id"]
        et  = entry.get("edit_type", "unknown")
        edit_type_counts[et] += 1

        # ── Missing images ─────────────────────────────────────────────
        src_path = raw_data_dir / entry["source_image"]
        tgt_path = raw_data_dir / entry["target_image"]
        if not (src_path.exists() and src_path.stat().st_size > 0):
            missing_src_ids.append(sid)
        if not (tgt_path.exists() and tgt_path.stat().st_size > 0):
            missing_tgt_ids.append(sid)

        # ── RLE mask validity (only when masks were loaded) ─────────────
        mask_was_loaded = entry.get("mask_loaded", False)
        annotations     = []

        if not mask_was_loaded:
            masks_not_loaded += 1
            # Defer RLE check — masks not yet joined from ImgEdit_recap_mask
        else:
            seg_path = raw_data_dir / entry["segmentation"]
            rle_valid = False

            if seg_path.exists():
                try:
                    seg_data = load_json(seg_path)
                    annotations = seg_data.get("annotations", [])
                    for ann in annotations:
                        rle = ann.get("segmentation")
                        if rle is None and "counts" in ann and "size" in ann:
                            rle = {"counts": ann["counts"], "size": ann["size"]}
                        if rle and isinstance(rle, dict):
                            mask = decode_rle_mask(rle)
                            if mask is not None and mask.sum() > 0:
                                rle_valid = True
                                break
                except Exception as e:
                    log.debug(f"seg parse error for {sid}: {e}")

            if not rle_valid:
                invalid_rle_ids.append(sid)

        # ── Bbox routing fallback ──────────────────────────────────────
        W = entry.get("img_width", 0)
        H = entry.get("img_height", 0)
        instruction = entry.get("edit_instruction", "")
        routed = route_bbox(et, instruction, annotations, W or 1, H or 1)
        if routed == [0, 0, 1000, 1000]:
            fallback_by_type[et] += 1
            if len(fallback_ids_by_type[et]) < 20:
                fallback_ids_by_type[et].append(sid)

    # ── Gates ─────────────────────────────────────────────────────────────
    gate_messages = []
    masks_loaded_count = total - masks_not_loaded

    if masks_loaded_count > 0:
        invalid_rle_frac = len(invalid_rle_ids) / max(masks_loaded_count, 1)
        if invalid_rle_frac > CFG.max_invalid_rle_frac:
            gate_messages.append(
                f"GATE FAIL — invalid_rle_masks {len(invalid_rle_ids):,}/{masks_loaded_count:,} "
                f"({invalid_rle_frac:.1%}) > {CFG.max_invalid_rle_frac:.1%}. Inspect before Phase 2."
            )
    else:
        invalid_rle_frac = 0.0
        gate_messages.append(
            "GATE DEFERRED — no masks loaded yet (load_masks=False in download). "
            "Re-run download with load_masks=True and audit again before Phase 1."
        )

    for et in ("add", "remove", "replace"):
        fb_count = fallback_by_type.get(et, 0)
        et_total = edit_type_counts.get(et, 0)
        fb_frac  = fb_count / max(et_total, 1)
        if et_total > 0 and fb_frac > CFG.max_fallback_frac:
            gate_messages.append(
                f"GATE FAIL — fallback rate for {et!r}: "
                f"{fb_count}/{et_total} ({fb_frac:.1%}) > {CFG.max_fallback_frac:.1%}. "
                f"Stage 1 supervision will be noisy."
            )

    gate_passed = len(gate_messages) == 0 or (
        len(gate_messages) == 1 and "DEFERRED" in gate_messages[0]
    )

    report = {
        "total_samples":              total,
        "missing_source_images":      len(missing_src_ids),
        "missing_source_image_ids":   missing_src_ids[:50],
        "missing_target_images":      len(missing_tgt_ids),
        "missing_target_image_ids":   missing_tgt_ids[:50],
        "masks_not_loaded":           masks_not_loaded,
        "invalid_rle_masks":          len(invalid_rle_ids),
        "invalid_rle_mask_ids":       invalid_rle_ids[:50],
        "invalid_rle_frac":           round(invalid_rle_frac, 4),
        "fallback_to_full_image_by_type": {
            et: {
                "count":     fallback_by_type.get(et, 0),
                "total":     edit_type_counts.get(et, 0),
                "fraction":  round(fallback_by_type.get(et, 0) / max(edit_type_counts.get(et, 1), 1), 4),
                "sample_ids": fallback_ids_by_type.get(et, []),
            }
            for et in sorted(fallback_by_type.keys())
        },
        "edit_type_distribution":     dict(sorted(edit_type_counts.items())),
        "_gate_passed":               gate_passed,
        "_gate_messages":             gate_messages,
    }

    save_json(report, CFG.audit_path)
    log.info(f"Audit report saved: {CFG.audit_path}")

    # ── Print summary ──────────────────────────────────────────────────────
    print(f"\n{'='*64}")
    print(f"AUDIT REPORT — {total:,} samples (v2)")
    print(f"{'='*64}")
    print(f"  missing source images : {report['missing_source_images']:,}")
    print(f"  missing target images : {report['missing_target_images']:,}")
    print(f"  masks not loaded      : {masks_not_loaded:,}  (load_masks=False in download)")
    if masks_loaded_count > 0:
        print(f"  invalid RLE masks     : {len(invalid_rle_ids):,} / {masks_loaded_count:,}  ({invalid_rle_frac:.1%})")
    else:
        print(f"  invalid RLE masks     : N/A (no masks loaded yet)")
    print()
    print("  Edit-type distribution:")
    for et, cnt in report["edit_type_distribution"].items():
        flag = "  ← NOT IN SPEC" if et not in {"add","remove","replace","adjust","style","background"} else ""
        print(f"    {et:22s}: {cnt:6,}{flag}")
    print()
    print("  Fallback-to-full-image by type:")
    for et, info in report["fallback_to_full_image_by_type"].items():
        flag = " ← GATE FAIL" if (
            info["fraction"] > CFG.max_fallback_frac and et in ("add","remove","replace")
        ) else ""
        print(f"    {et:22s}: {info['count']:5,}/{info['total']:5,}  ({info['fraction']:.1%}){flag}")
    print()

    for msg in gate_messages:
        prefix = "  ⚠ " if "DEFERRED" in msg else "  ✗ "
        print(f"{prefix}{msg}")
    if gate_passed and not gate_messages:
        print("  ✓  All quality gates PASSED.")
    print(f"{'='*64}\n")
    return report


print("audit_dataset() defined (v2).")


## §6 — Smoke Test (Tiny Subset)
Download **50 samples** and run a full audit. Verifies the entire §4–§5 pipeline  
end-to-end on minimal data before committing to the full 50K download.  
**Expected runtime: ~3–5 min** (streaming 50 samples + image saves + audit).


In [ ]:
# ── §6.1  Smoke test — 5 samples, metadata only (no masks yet) ────────────
# Tests: parquet enumeration, edit_type inference, image fetch via HfFileSystem,
# manifest structure, and audit on tiny subset.
# load_masks=False — smoke test focuses on parquet + image fetch + audit.
# Tar-based image fetching is now the primary strategy (§6.1 fix).
# Expected runtime: ~2–5 min (5 HF image fetches + audit).

SMOKE_N   = 10
SMOKE_DIR = CFG.drive_root / "data" / "smoke_test_v2"

print(f"Smoke test: n={SMOKE_N}, output={SMOKE_DIR}")
print("Running with load_masks=False (focus: parquet + tar image fetch + audit).")
print("REQUIRES §2.5 has been run to populate image tar name index.")
print()

# ── Step 1: download ────────────────────────────────────────────────────
smoke_manifest_path = download_imgedit_subset(
    n=SMOKE_N,
    output_dir=SMOKE_DIR,
    load_masks=True,
    force=False,
    edit_type_filter = ["background"]
)
smoke_manifest = load_json(smoke_manifest_path)
print(f"\nSaved {len(smoke_manifest)} samples.")

# ── Step 2: verify manifest schema ─────────────────────────────────────
required_keys = {
    "sample_id", "hf_sample_id", "pq_file", "edit_type",
    "source_image", "target_image", "segmentation",
    "edit_instruction", "img_width", "img_height",
    "mask_loaded", "shard_id", "row_index", "mask_path",
}
for entry in smoke_manifest:
    missing = required_keys - set(entry.keys())
    assert not missing, f"Entry {entry['sample_id']} missing keys: {missing}"
print("✓ All manifest entries have required keys.")

# ── Step 3: check sample_id extraction ─────────────────────────────────
for entry in smoke_manifest:
    sid = entry["hf_sample_id"]
    assert sid and sid != "UNKNOWN", f"hf_sample_id extraction failed: {sid!r}"
print(f"✓ hf_sample_id extracted correctly (e.g. {smoke_manifest[0]['hf_sample_id']!r})")

# ── Step 4: check edit_type inference ──────────────────────────────────
edit_types_seen = set(e["edit_type"] for e in smoke_manifest)
unknown = edit_types_seen - KNOWN_EDIT_TYPES - {"unknown"}
if "unknown" in edit_types_seen:
    print(f"WARNING: some samples have edit_type='unknown'. Check parquet filenames.")
else:
    print(f"✓ edit_types all known: {sorted(edit_types_seen)}")

# ── Step 5: verify image files ──────────────────────────────────────────
from PIL import Image as PILImage
n_ok, n_bad = 0, 0
for entry in smoke_manifest:
    for key in ["source_image", "target_image"]:
        p = SMOKE_DIR / entry[key]
        try:
            PILImage.open(p).verify()
            n_ok += 1
        except Exception as e:
            print(f"  CORRUPT: {p}: {e}")
            n_bad += 1
if n_bad > 0:
    print(f"WARNING: {n_bad} corrupt images. HfFileSystem may need tar extraction instead.")
    print("  If all images failed: the HF repo stores images in tar archives.")
    print("  See §4.1 fetch_image_from_hf() — add tar fallback if needed.")
else:
    print(f"✓ {n_ok} image files verified readable.")

# ── Step 6: audit ───────────────────────────────────────────────────────
print("\nRunning audit (masks not loaded — gate will be deferred)...")
report = audit_dataset(raw_data_dir=SMOKE_DIR)

assert report["total_samples"] == len(smoke_manifest)
# When load_masks=False, gate is deferred; with load_masks=True gate runs fully.
if not any("DEFERRED" in m for m in report["_gate_messages"]):
    print("✓ Audit ran with masks loaded — gate fully evaluated.")
    assert report["_gate_passed"], f"Gate failed: {report['_gate_messages']}"
else:
    print("✓ Audit structure valid (masks not loaded — gate deferred).")
    print("  Re-run with load_masks=True for full RLE gate.")

print()
print("=" * 64)
print("SMOKE TEST PASSED — Phase 0 v2 pipeline is functional.")
print()
print("NEXT STEPS:")
print("  1. ✓ §2.3 complete — MASK_INNER_RLE_FIELD=\'mask\', MASK_BOX_FORMAT=\'xyxy\'")
print("  2. ✓ §6.1 fixes applied — parquet, path extraction, tar image fetch")
print("  3. Once smoke passes: re-run with load_masks=True to verify mask join")
print("  4. Run §7.1 full download (n=50_000, load_masks=True)")
print("=" * 64)


## §7 — Full Dataset Download (50K)
*Run only after the smoke test passes. Estimated ~30–90 min depending on network.*

In [ ]:
# ── §7.1  Full download — 50K samples (run after smoke test passes) ────────
# Re-runnable: manifest check inside download_imgedit_subset will skip if done.
# Set force=True to re-download.
#
# §2.3 COMPLETE: MASK_INNER_RLE_FIELD='mask', MASK_BOX_FORMAT='xyxy' confirmed.
# §2.4 constants are set — safe to run Stage B with load_masks=True.
#
# Stage A: metadata + images only (skip if already done)
# Stage B: re-run with load_masks=True (READY — §2.3 confirmed mask schema)

STAGE = "B"   # change to "B" after §2.3 is done and mask fields are set

print(f"Starting full download Stage {STAGE}: n=10000 samples")
print(f"  load_masks = {STAGE == 'B'}")
print(f"  Output dir : {CFG.data_dir}")
print()

manifest_path = download_imgedit_subset(
    n          = CFG.n_subset,
    output_dir = CFG.data_dir,
    load_masks = (STAGE == "B"),
    force      = False,
    edit_type_filter = ["background", "adjust"]
)

manifest = load_json(manifest_path)
print(f"\n✓ Download complete: {len(manifest):,} samples")
print(f"  Manifest: {manifest_path}")
if STAGE == "A":
    print()
    print("  Stage A done. §2.3 + §6.1 fixes confirmed.")
    print("  Run §6.1 smoke test with load_masks=True to verify mask join.")
    print("  Then change STAGE = \"B\" here and re-run for full 50K download with masks.")


In [ ]:
# ── §7.2  Full dataset audit — run after §7.1 ─────────────────────────────
# Estimated runtime: ~5–15 min for 50K samples.
# This is a GATE: do not proceed to Phase 1 until _gate_passed is True.

print("Running full dataset audit...")
report = audit_dataset(raw_data_dir=CFG.data_dir)

if not report["_gate_passed"]:
    print()
    print("STOP: Quality gate failed. Resolve issues above before Phase 1.")
    print("Relevant files for inspection:")
    print(f"  {CFG.audit_path}")
    print(f"  {CFG.data_dir / 'download_errors.json'}")
else:
    print()
    print("Quality gates passed. Proceed to Phase 1 (VLM fine-tuning).")


## §7.3 — Subset Manifest: Mask-Loaded, Real-Bbox Samples Only
Filters `samples.json` to a clean training subset by removing two categories of entries  
that would degrade inpainting quality:

1. **`mask_loaded=False`** — no RLE mask was joined from `ImgEdit_recap_mask`; no  
   localised segmentation supervision is available for Phase 2.
2. **bbox fallback = full image** — `route_bbox()` returned `[0, 0, 1000, 1000]`,  
   meaning no valid annotation bbox exists. Includes all `style` / `background` entries  
   (always-global by design) and any `add` / `remove` / `replace` sample where the mask  
   dataset had no object annotation.

Output: `samples_filtered.json` — use this for Phase 1 and Phase 2 training.  
`samples.json` is **preserved unchanged**; re-run this cell freely with different criteria.


In [ ]:
# -- §7.3  Subset manifest -- keep only mask-loaded, non-fallback samples ----
# Purpose: produce a clean training subset from the downloaded samples by
# removing two categories of entries that cannot be used for localised
# inpainting training:
#
#   1. mask_loaded=False  -- the RLE mask join from ImgEdit_recap_mask failed.
#      These samples have no segmentation supervision available for Phase 2.
#      Root cause: either the sample_id join key did not match, or the sample
#      originates from a partition not covered by the mask dataset (e.g. action
#      video-frame splits).
#
#   2. bbox_fallback=True -- route_bbox() returned the full-image sentinel
#      [0, 0, 1000, 1000].  This happens when:
#        (a) edit_type is always-global (style, background)
#        (b) an adjust edit matched a global-adjust keyword
#        (c) no valid annotation bbox was found in the segmentation JSON
#      Full-image conditioning is useless for localised inpainting -- the
#      entire image is the 'region of interest', so the model receives no
#      spatial supervision signal.
#
# This cell is RE-RUNNABLE: reads samples.json (unchanged) and writes
# samples_filtered.json.  Re-run freely to experiment with filter criteria.
# Downstream training cells (Phase 1 build_vlm_dataset, Phase 2 cache step)
# should load samples_filtered.json instead of samples.json.

from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

# Full-image sentinel value as returned by route_bbox() for fallback cases.
# Defined here so any future change to the sentinel value is caught at filter
# time (a mismatch would cause zero samples to be removed, not silent misfilter).
_FULL_IMAGE_SENTINEL = [0, 0, 1000, 1000]


def subset_manifest_clean(
    raw_data_dir=None,
    output_filename="samples_filtered.json",
):
    """Filter samples.json to entries with a valid mask AND a real edit bbox.

    Two-pass filtering (order matters for accurate diagnostics):

      Pass 1 -- mask_loaded check:
        Discard samples where mask_loaded=False.  Including them would force
        Phase 2 to use a full-image fallback mask, corrupting inpainting
        supervision.  Checked first because it requires no disk I/O.

      Pass 2 -- bbox routing check:
        Reload each remaining sample's segmentation JSON and call route_bbox()
        with the SAME logic used by audit_dataset() (no reimplementation here).
        Discard samples where route_bbox() returns [0, 0, 1000, 1000].
        Consistency with audit_dataset() is intentional: any sample that passes
        the audit gate also passes this filter, and vice versa.

    Args:
        raw_data_dir:    Path to directory containing samples.json and the
                         segmentation/ subdirectory. Defaults to CFG.data_dir.
        output_filename: Filename for the filtered manifest (written to raw_data_dir).

    Returns:
        Path to the written filtered manifest (samples_filtered.json by default).

    Raises:
        AssertionError: if 0 samples survive filtering (likely masks not loaded
                        -- check that §7.1 was run with STAGE='B').
        AssertionError: if kept + removed counts do not sum to total (logic bug).
    """
    if raw_data_dir is None:
        raw_data_dir = CFG.data_dir
    raw_data_dir  = Path(raw_data_dir)
    manifest_path = raw_data_dir / "samples.json"
    filtered_path = raw_data_dir / output_filename
    manifest      = load_json(manifest_path)
    total         = len(manifest)
    log.info("subset_manifest_clean: loaded %d samples from %s", total, manifest_path)

    kept             = []
    removed_no_mask  = []   # reason: mask_loaded=False
    removed_fallback = []   # reason: route_bbox returned full-image sentinel
    n_seg_errors     = 0    # seg JSON load failures (counted inside removed_fallback)

    for entry in tqdm(manifest, desc="Filtering samples", unit="sample"):
        sid = entry["sample_id"]

        # -- Filter 1: mask must have been successfully loaded ---------------
        # mask_loaded=True means download_imgedit_subset() found a non-empty
        # annotations list from ImgEdit_recap_mask and saved it to the seg JSON.
        if not entry.get("mask_loaded", False):
            removed_no_mask.append(sid)
            continue

        # -- Filter 2: bbox routing must not fall back to full image ---------
        # Load annotations from the segmentation JSON on disk (same source as
        # audit_dataset) and call route_bbox() -- the single authoritative
        # routing implementation from §3.1.  We do NOT reimplement the routing
        # logic here to guarantee consistency with the audit gate.
        annotations = []
        try:
            seg_path = raw_data_dir / entry["segmentation"]
            if seg_path.exists() and seg_path.stat().st_size > 0:
                seg_data    = load_json(seg_path)
                annotations = seg_data.get("annotations", [])
            else:
                # Missing or empty seg file -- treat as fallback (conservative).
                # This is a data integrity issue; log and exclude.
                log.warning(
                    "Seg file missing/empty for %s: %s -- removing as fallback",
                    sid, entry["segmentation"]
                )
                n_seg_errors += 1
                removed_fallback.append(sid)
                continue
        except Exception as exc:
            log.warning("Seg JSON load error for %s: %s -- removing as fallback", sid, exc)
            n_seg_errors += 1
            removed_fallback.append(sid)
            continue

        et          = entry.get("edit_type", "unknown")
        instruction = entry.get("edit_instruction", "") or ""
        # img_width/height may be 0 for rare parse failures; guard with max(,1)
        W = max(entry.get("img_width",  1) or 1, 1)
        H = max(entry.get("img_height", 1) or 1, 1)

        routed = route_bbox(et, instruction, annotations, W, H)

        if routed == _FULL_IMAGE_SENTINEL:
            removed_fallback.append(sid)
            continue

        # -- Sample passed both filters -------------------------------------
        kept.append(entry)

    # -- Sanity assertions before saving -----------------------------------
    n_kept     = len(kept)
    n_no_mask  = len(removed_no_mask)
    n_fallback = len(removed_fallback)
    n_removed  = n_no_mask + n_fallback

    assert n_kept + n_no_mask + n_fallback == total, (
        "Count invariant violated: {} kept + {} no_mask + {} fallback = {} \!= {}".format(
            n_kept, n_no_mask, n_fallback, n_kept + n_no_mask + n_fallback, total
        )
    )
    assert n_kept > 0, (
        "All {} samples were removed.  Check:\n"
        "  (a) Was §7.1 run with STAGE='B' (load_masks=True)?\n"
        "      mask_loaded=False on every sample means masks were never joined.\n"
        "  (b) Does samples.json exist at {}?".format(total, manifest_path)
    )

    # -- Persist filtered manifest ----------------------------------------
    save_json(kept, filtered_path)
    log.info("Saved filtered manifest: %s (%d samples)", filtered_path, n_kept)

    # -- Build per-type breakdown tables ----------------------------------
    kept_types       = Counter(e["edit_type"] for e in kept)
    input_by_type    = Counter(e["edit_type"] for e in manifest)
    removed_nm_set   = set(removed_no_mask)
    removed_fb_set   = set(removed_fallback)
    removed_nm_types = Counter(
        e["edit_type"] for e in manifest if e["sample_id"] in removed_nm_set
    )
    removed_fb_types = Counter(
        e["edit_type"] for e in manifest if e["sample_id"] in removed_fb_set
    )

    # -- Print report -------------------------------------------------------
    sep = "=" * 72
    print("\n" + sep)
    print("MANIFEST SUBSET REPORT  (§7.3)")
    print(sep)
    print("  Input  : {:>7,}  samples  ->  {}".format(total, manifest_path.name))
    print("  Kept   : {:>7,}  samples  ->  {}".format(n_kept, filtered_path.name))
    print("  Removed: {:>7,}  samples  ({:.1%} of input)".format(
        n_removed, n_removed / max(total, 1)))
    print()
    print("  Removal breakdown:")
    print("    mask not loaded  (mask_loaded=False) : {:>6,}  ({:.1%})".format(
        n_no_mask, n_no_mask / max(total, 1)))
    print("    bbox fallback to full image          : {:>6,}  ({:.1%})".format(
        n_fallback, n_fallback / max(total, 1)))
    if n_seg_errors:
        print("      of which: seg-file load errors    : {:>6,}".format(n_seg_errors))
    print()
    print("  Per-type breakdown:")
    hdr = "  {:<22}  {:>6}  {:>6}  {:>6}  {:>12}  {:>12}"
    print(hdr.format("edit_type", "kept", "input", "%kept", "rm_no_mask", "rm_fallback"))
    print("  " + "-" * 70)
    row_fmt = "  {:<22}  {:>6}  {:>6}  {:>6}  {:>12}  {:>12}{}"
    for et in sorted(input_by_type.keys()):
        n_in  = input_by_type[et]
        n_k   = kept_types.get(et, 0)
        n_nm  = removed_nm_types.get(et, 0)
        n_fb  = removed_fb_types.get(et, 0)
        pct   = "{:.0%}".format(n_k / max(n_in, 1))
        flag  = "  <- low yield" if n_k / max(n_in, 1) < 0.30 else ""
        print(row_fmt.format(et, "{:,}".format(n_k), "{:,}".format(n_in), pct,
                             "{:,}".format(n_nm), "{:,}".format(n_fb), flag))
    print()
    print("  Filtered manifest saved: {}".format(filtered_path))
    print()
    print("  NEXT STEPS:")
    print("  Phase 1 build_vlm_dataset()       : pass manifest_path=filtered_path")
    print("  Phase 2 cache_vlm_hidden_states() : pass manifest_path=filtered_path")
    print(sep + "\n")

    return filtered_path


# -- Run the filter -------------------------------------------------------
filtered_manifest_path = subset_manifest_clean(raw_data_dir=CFG.data_dir)
filtered_manifest      = load_json(filtered_manifest_path)

# Post-filter invariant: every kept sample must have mask_loaded=True.
# Rechecked here cheaply (no seg file reload) as a guard against logic bugs.
bad_sids = [e["sample_id"] for e in filtered_manifest if not e.get("mask_loaded", False)]
assert not bad_sids, (
    "Post-filter invariant violated: {} samples have mask_loaded=False. "
    "First offenders: {}".format(len(bad_sids), bad_sids[:5])
)
print("Post-filter assertion passed: all {:,} kept samples have mask_loaded=True".format(
    len(filtered_manifest)))
print("Filtered manifest path: {}".format(filtered_manifest_path))
